In [66]:
import requests as rq
from curl_cffi import requests as rq2
from selenium import webdriver
from bs4 import BeautifulSoup as bs
import pandas as pd
from time import sleep
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import Select
from pypdf import PdfReader
import io
import re
from google import genai

In [67]:
url = 'https://www.topuniversities.com/latin-america-south-america-rankings/2025?countries=pe'
driver = webdriver.Chrome()
driver.implicitly_wait(60)
driver.set_page_load_timeout(60)
driver.get(url)
driver.maximize_window()
sleep(3)
html = driver.page_source
driver.quit()

In [68]:
soup = bs(html)
universidades = []
nombres = soup.find_all('a', {'class': 'uni-link'}, href=True)
for nombre in nombres:
    universidades.append(nombre.text.strip())
print(universidades)
universidades = universidades[:15]
print(universidades)
df_universidad = pd.DataFrame(universidades, columns=['universidad'])
df_universidad['id_universidad'] = range(1, len(df_universidad) + 1)
df_universidad = df_universidad[['id_universidad', 'universidad']]
df_universidad.head(15)
df_universidad.to_csv('top15_universidades.csv', index=False, encoding='utf-8-sig')


['Pontificia Universidad Catolica del Peru (PUCP)', 'Universidad Nacional Mayor de San Marcos', 'Universidad Peruana Cayetano Heredia (UPCH)', 'Universidad del Pacífico', 'Universidad de Lima', 'Universidad Peruana de Ciencias Aplicadas (UPC)', 'Universidad Nacional de Ingeniería', 'Universidad Nacional Agraria la Molina', 'Universidad San Ignacio de Loyola (USIL)', 'Universidad de Piura', 'Universidad Científica del Sur', 'Universidad ESAN', 'Universidad de San Martín de Porres', 'Universidad Nacional de San Agustín de Arequipa', 'Universidad Nacional de Trujillo', 'Universidad Ricardo Palma', 'Universidad Católica San Pablo', 'Universidad Norbert Wiener', 'Universidad Privada del Norte', 'Universidad Nacional Federico Villarreal (UNFV)', 'Universidad Nacional de San Antonio Abad del Cusco', 'Universidad Nacional del Altiplano', 'Universidad Continental', 'Universidad César Vallejo']
['Pontificia Universidad Catolica del Peru (PUCP)', 'Universidad Nacional Mayor de San Marcos', 'Unive

In [69]:
df_universidad.to_csv('top15_universidades.csv', index=False, encoding='utf-8-sig')

In [70]:
df_carreras = pd.DataFrame(columns=['id_carrera', 'id_universidad', 'carrera', 'descripcion'])
df_cursos = pd.DataFrame(columns=['id_curso', 'id_universidad', 'id_carrera', 'curso', 'ciclo'])

# **1. PUCP**

In [71]:
import time
from selenium import webdriver

url_pucp = 'https://estudios-generales-ciencias.pucp.edu.pe/informacion-academica/plan-de-estudios/'

driver = webdriver.Chrome()
driver.implicitly_wait(60)
driver.set_page_load_timeout(60)
driver.get(url_pucp)
driver.maximize_window()
sleep(3)

xpath_inge_informatica_pucp = '//*[@id="contenedor-eventos"]/div[10]/div/div[2]/a[1]'
driver.find_element(by='xpath', value=xpath_inge_informatica_pucp).click()
sleep(3) 
html_informatica_pucp = driver.page_source

driver.get(url_pucp)
sleep(3)

xpath_inge_telecomunicaciones_pucp = '//*[@id="contenedor-eventos"]/div[5]/div/div[2]/a[1]' 
driver.find_element(by='xpath', value=xpath_inge_telecomunicaciones_pucp).click()
time.sleep(3)
html_telecomunicaciones_pucp = driver.page_source

sleep(3)
driver.quit()

In [72]:
soups_pucp = [bs(html_informatica_pucp), bs(html_telecomunicaciones_pucp)]
id_carrera_actual_pucp = 1

for soup in soups_pucp:
    carrera = soup.find('h1', {'class': 'heading l'}).text.strip()
    primer_parrafo = soup.find('p')
    descripcion = primer_parrafo.text.strip() if primer_parrafo else ""
    nueva_fila = {
        'id_universidad': 1, 
        'id_carrera': id_carrera_actual_pucp,
        'carrera': carrera,
        'descripcion': descripcion
    }
    
    df_carreras = pd.concat([df_carreras, pd.DataFrame([nueva_fila])], ignore_index=True)
    id_carrera_actual_pucp += 1

df_carreras

,id_carrera,id_universidad,carrera,descripcion
0,1,1,Ingeniería Informática,La ingeniería informática se encarga de desarr...
1,2,1,Ingeniería de las Telecomunicaciones,Las telecomunicaciones cubren la necesidad de ...


In [73]:
id_carrera_actual_pucp = 1

for soup in soups_pucp:
    niveles = soup.find_all('div', {'class': 'div-requisito'})

    for nivel_div in niveles:
        nivel_texto = nivel_div.find('h5', {'class': 's-p-b-2'}).text.strip()
        nivel = nivel_texto.replace('Nivel', '').strip() 
        cursos_nivel = nivel_div.find_all('div', {'class': 'flex-requisito'})

        for curso_div in cursos_nivel:
            curso_tag = curso_div.find('a', {'class': 'link-item-curso'})
            if curso_tag is None:
                continue 
            nombre_curso = curso_tag.text.strip()
            nueva_fila = {
                'id_curso': len(df_cursos) + 1,
                'id_universidad': 1,
                'id_carrera': id_carrera_actual_pucp,
                'curso': nombre_curso,
                'ciclo': nivel
            }

            df_cursos = pd.concat([df_cursos, pd.DataFrame([nueva_fila])], ignore_index=True)

    id_carrera_actual_pucp += 1

# **SAN MARCOS**

In [74]:
url_sanmarcos = 'https://www.unmsm.edu.pe/formacion-academica/carreras-de-pregrado'

driver = webdriver.Chrome()
driver.implicitly_wait(60)
driver.set_page_load_timeout(60)
driver.get(url_sanmarcos)
driver.maximize_window()
sleep(3)

# ==============================================================================
# #1 COMPUTACIÓN CIENTÍFICA
# ==============================================================================
selector = Select(driver.find_element(by='id', value='select-pregrado-areas'))
selector.select_by_visible_text('Ciencias Básicas')
sleep(3)

xpath_compu_cientifica_sanmarcos = '//*[@id="list-carreras-pregrado"]/div[2]/div/a[2]/div/p[1]'
driver.find_element(by='xpath', value=xpath_compu_cientifica_sanmarcos).click()
sleep(3)
html_compu_cientifica_sanmarcos = driver.page_source
sleep(3)

# ==============================================================================
# #2 CIENCIAS DE LA COMPUTACIÓN
# ==============================================================================
driver.get(url_sanmarcos)
sleep(3)

selector = Select(driver.find_element(by='id', value='select-pregrado-areas'))
selector.select_by_visible_text('Ingenierías')
sleep(3)

xpath_ciencias_compu_sanmarcos = '//*[@id="list-carreras-pregrado"]/div[2]/div/a[2]/div/p[1]'
driver.find_element(by='xpath', value=xpath_ciencias_compu_sanmarcos).click()
sleep(3)
html_ciencias_compu_sanmarcos = driver.page_source
sleep(3)

# ==============================================================================
# #3 INGENIERÍA DE SISTEMAS, SOFTWARE Y TELECOMUNICACIONES (PÁGINA 2)
# ==============================================================================
# --- Procesamiento de Ingeniería de Sistemas ---
driver.get(url_sanmarcos)
sleep(3)
selector = Select(driver.find_element(by='id', value='select-pregrado-areas'))
selector.select_by_visible_text('Ingenierías')
sleep(2)
driver.find_element(by='id', value='index-2').click()  # Ir a la página 2
sleep(3)

xpath_inge_sistemas_sanmarcos = '//*[@id="list-carreras-pregrado"]/div[1]/div/a[2]/div/p[1]'
driver.find_element(by='xpath', value=xpath_inge_sistemas_sanmarcos).click()
sleep(3)
html_inge_sistemas_sanmarcos = driver.page_source

# --- Procesamiento de Ingeniería de Software ---
driver.get(url_sanmarcos)
sleep(3)
selector = Select(driver.find_element(by='id', value='select-pregrado-areas'))
selector.select_by_visible_text('Ingenierías')
sleep(2)
driver.find_element(by='id', value='index-2').click()  # Ir a la página 2
sleep(3)

xpath_inge_software_sanmarcos = '//*[@id="list-carreras-pregrado"]/div[2]/div/a[2]/div/p[1]'
driver.find_element(by='xpath', value=xpath_inge_software_sanmarcos).click()
sleep(3)
html_inge_software_sanmarcos = driver.page_source

# Finalizar
sleep(3)
driver.quit()

In [75]:
soups_sanmarcos = [bs(html_compu_cientifica_sanmarcos),
                   bs(html_ciencias_compu_sanmarcos),
                   bs(html_inge_sistemas_sanmarcos),
                   bs(html_inge_software_sanmarcos)]

In [76]:
id_carrera_actual_sanmarcos = 1

for soup in soups_sanmarcos:
    carrera = soup.find('p', {'class': 'h3 color-white font-weight-bold'}).text
    descripcion = soup.find('p', {'class': 'size-1-1rem mb-0'}).text
    nueva_fila = {
        'id_universidad': 2, 
        'id_carrera': id_carrera_actual_sanmarcos,
        'carrera': carrera,
        'descripcion': descripcion
    }
    
    df_carreras = pd.concat([df_carreras, pd.DataFrame([nueva_fila])], ignore_index=True)
    id_carrera_actual_sanmarcos += 1

df_carreras

,id_carrera,id_universidad,carrera,descripcion
0,1,1,Ingeniería Informática,La ingeniería informática se encarga de desarr...
1,2,1,Ingeniería de las Telecomunicaciones,Las telecomunicaciones cubren la necesidad de ...
2,1,2,Computación Científica,La Escuela Profesional de Computación Científi...
3,2,2,Ciencia de la Computación,La carrera de Ciencia de la Computación forma ...
4,3,2,Ingeniería de Sistemas,La Escuela Profesional de Ingeniería de Sistem...
5,4,2,Ingeniería de Software,La Escuela Profesional de Ingeniería de Softwa...


In [77]:
mallas_pdf_sanmarcos = []
for soup in soups_sanmarcos:
    pdf = soup.find('div', {'id':'carrera-info'}).find('a', href=True)['href']
    if pdf:
        mallas_pdf_sanmarcos.append(pdf)
    else:
        mallas_pdf_sanmarcos.append('no hay')
mallas_pdf_sanmarcos

['https://matematicas.unmsm.edu.pe/img/about/ep-computacion-cientifica/Plan%20de%20estudios%202018.pdf',
 'https://biologia-web-unmsm-v3.s3.us-east-2.amazonaws.com/Plan_Estudios_2023_CIENCIA_DE_LA_COMPUTACION_9f48fa6285.pdf',
 'https://biologia-web-unmsm-v3.s3.us-east-2.amazonaws.com/Plan_Estudios_2023_INGENIERIA_DE_SISTEMAS_3a6a8d123a.pdf',
 'https://biologia-web-unmsm-v3.s3.us-east-2.amazonaws.com/Plan_Estudios_2023_INGENIERIA_DE_SOFTWARE_155598bb79.pdf']

In [169]:
#['https://matematicas.unmsm.edu.pe/img/about/ep-computacion-cientifica/Plan%20de%20estudios%202018.pdf',
#'https://biologia-web-unmsm-v3.s3.us-east-2.amazonaws.com/Plan_Estudios_2023_CIENCIA_DE_LA_COMPUTACION_9f48fa6285.pdf',
#'https://biologia-web-unmsm-v3.s3.us-east-2.amazonaws.com/Plan_Estudios_2023_INGENIERIA_DE_SISTEMAS_3a6a8d123a.pdf',
#'https://biologia-web-unmsm-v3.s3.us-east-2.amazonaws.com/Plan_Estudios_2023_INGENIERIA_DE_SOFTWARE_155598bb79.pdf']

ID_UNIV_SANMARCOS = 3

carreras_sanmarcos = [
    ('Plan de estudios 2018 (1).pdf', 'Computación Científica', 'narrativo'),
    ('Plan_Estudios_2023_CIENCIA_DE_LA_COMPUTACION_9f48fa6285 (1).pdf', 'Ciencia de la Computación', 'tabular'),
    ('Plan_Estudios_2023_INGENIERIA_DE_SISTEMAS_3a6a8d123a (1).pdf', 'Ingeniería de Sistemas', 'tabular'),
    ('Plan_Estudios_2023_INGENIERIA_DE_SOFTWARE_155598bb79 (1).pdf', 'Ingeniería de Software', 'tabular'),
]

for archivo, nombre_carrera, tipo in carreras_sanmarcos:
    ruta = f'C:/Users/Mildred/Downloads/{archivo}'
    if tipo == 'tabular':
        cursos_extraidos = parsear_malla_tabular(ruta) 
        for ciclo, codigo, nombre, creditos in cursos_extraidos:
            nueva_fila = {
                'universidad': 'Universidad Nacional Mayor de San Marcos',
                'id_carrera': nombre_carrera,
                'curso': nombre,
                'ciclo': ciclo
            }
            df_cursos = pd.concat([df_cursos, pd.DataFrame([nueva_fila])], ignore_index=True)

    elif tipo == 'narrativo':
        cursos_extraidos = parsear_malla_2018(ruta)
        for ciclo, nombre, creditos in cursos_extraidos:
            nueva_fila = {
                'universidad': 'Universidad Nacional Mayor de San Marcos',
                'id_carrera': nombre_carrera,
                'curso': nombre,
                'ciclo': ciclo
            }
            df_sanmarcos = pd.concat([df, pd.DataFrame([nueva_fila])], ignore_index=True)

df_sanmarcos

NameError: name 'parsear_malla_2018' is not defined

# **CAYETANO HEREDIA**

In [81]:

#Link: https://drive.google.com/file/d/1ZMWsnPJFGXOzAncEvUl0MBbzl-AKyoLZ/view
#Link: https://drive.google.com/file/d/1-LmMkggFD3egiMPP6RpXWfCQ7z5LQ627/view?usp=sharing
#Link: https://drive.google.com/file/d/1-LmMkggFD3egiMPP6RpXWfCQ7z5LQ627/view?usp=sharing
#Link: https://drive.google.com/file/d/1ZMWsnPJFGXOzAncEvUl0MBbzl-AKyoLZ/view
#Link: https://drive.google.com/file/d/1ZMWsnPJFGXOzAncEvUl0MBbzl-AKyoLZ/view
#Link: https://drive.google.com/file/d/1B17AO8XR7KVErqdjhwPTGox6e6MpFkIt/view?usp=drive_link
#Link: https://drive.google.com/file/d/1PvkB_XbA8Y_WBQ9IKJLiHY2jWNaUXwpd/view?usp=drive_link
#Link: https://drive.google.com/file/d/1nMFaGUL1xq7ONXhDEmyu7zmBbcxgB6Vd/view?usp=sharing

ID_UNIV_INFORMATICA = 3
ID_CARRERA_INFORMATICA = 1
 
CICLO_MAP_cayetano = {
    'PRIMER': 1, 'SEGUNDO': 2, 'TERCER': 3, 'CUARTO': 4, 'QUINTO': 5,
    'SEXTO': 6, 'SÉPTIMO': 7, 'SEPTIMO': 7, 'OCTAVO': 8, 'NOVENO': 9,
    'DECIMO': 10, 'DÉCIMO': 10
}
 
 
def parsear_malla_informatica_2024(path):
    """
    Para 'PLAN DE ESTUDIOS DE INGENIERÍA INFORMÁTICA 2024.pdf'.
    Formato de tabla: CÓDIGO | NOMBRE | TIPO DE ESTUDIOS | MODALIDAD |
    TIPO DE ASIGNATURA | PRE-REQUISITO | 14 columnas numéricas
    (horas teóricas/prácticas P/V/T + THL + créditos P/V/T, la última
    columna es el TOTAL DE CRÉDITOS OTORGADOS).
    Solo toma el cuerpo de Ciclo 1 a Ciclo 10 (ignora el resumen final,
    la tabla de electivas detalladas y las actividades complementarias).
    Devuelve lista de tuplas (ciclo, codigo, nombre, creditos)
    """
    lector = PdfReader(path)
    texto = "\n".join(p.extract_text() for p in lector.pages)
 
    cuerpo = texto.split('RESUMEN DE HORAS')[0]
 
    partes = re.split(
        r'(PRIMER|SEGUNDO|TERCER|CUARTO|QUINTO|SEXTO|S[ÉE]PTIMO|OCTAVO|NOVENO|D[ÉE]CIMO)\s*CICLO',
        cuerpo
    )
 
    patron_fila = re.compile(
        r'([A-Z]{1,4}\d{2,5})\s+(.+?)\s+(GENERAL|ESPEC[ÍI]FICO|DE\s+ESPECIALIDAD)\s+'
        r'(PRESENCIAL|NO\s*PRESENCIAL|SEMI\s*PRESENCIAL|SEMIPRESENCIAL)\s+'
        r'(OBLIGATORIO|ELECTIVO|ELECTIV0)\s+'
        r'(.+?)\s+'
        r'((?:\d+\s+){13}\d+)'
    )
 
    resultados = []
    for i in range(1, len(partes), 2):
        ciclo = CICLO_MAP_cayetano[partes[i]]
        bloque = re.sub(r'\s+', ' ', partes[i + 1]).strip()
        bloque = bloque.split('STC')[0]
 
        for m in patron_fila.finditer(bloque):
            codigo = m.group(1)
            nombre = re.sub(r'\s+', ' ', m.group(2)).strip()
            numeros = m.group(7).split()
            creditos_totales = numeros[-1] 
            resultados.append((ciclo, codigo, nombre, creditos_totales))
 
    return resultados
 
ruta_informatica = "C:/Users/Mildred/Downloads/PLAN DE ESTUDIOS DE INGENIERÍA INFORMÁTICA  2024.pdf"
 
cursos_informatica = parsear_malla_informatica_2024(ruta_informatica) 
 
for ciclo, codigo, nombre, creditos in cursos_informatica:
    nueva_fila = {
        'id_curso': len(df_cursos) + 1,
        'id_universidad': ID_UNIV_INFORMATICA,
        'id_carrera': ID_CARRERA_INFORMATICA,
        'curso': nombre,
        'ciclo': ciclo
    }
    df_cursos = pd.concat([df_cursos, pd.DataFrame([nueva_fila])], ignore_index=True)
 
print(f'Cursos agregados: {len(cursos_informatica)}')
print(df_cursos[df_cursos['id_universidad'] == ID_UNIV_INFORMATICA].shape)
df_cursos

Cursos agregados: 68
(136, 5)


,id_curso,id_universidad,id_carrera,curso,ciclo
0,1,1,1,Álgebra matricial y geometría analítica,1
1,2,1,1,Comunicación académica,1
2,3,1,1,Fundamentos de cálculo,1
3,4,1,1,Fundamentos de física,1
4,5,1,1,Química 1,1
...,...,...,...,...,...
184,185,3,1,ASIGNATURA ELECTIVA III,9
185,186,3,1,TRABAJO DE INVESTIGACIÓN,10
186,187,3,1,DERECHO Y ÉTICA EN LA INFORMÁTICA,10
187,188,3,1,TÓPICOS AVANZADOS DE SEGURIDAD INFORMÁTICA,10


In [108]:
import pandas as pd

# 1. Primer cruce: Unir cursos con carreras usando 'id_carrera'
df_total = pd.merge(df_cursos, df_carreras, on='id_carrera', how='left')
print(df_total)
# 2. Segundo cruce: Unir el resultado con universidades.
# Asumimos que vas a cruzar 'id_universidad_x' del df_total con 'id_universidad' del df_universidad
df_total = pd.merge(df_total, df_universidad, left_on='id_universidad_x', right_on='id_universidad', how='left')

# 3. Finalmente, filtras las columnas que realmente necesitas para tu resultado final.
# Cambia 'nombre_universidad' por el nombre real de la columna en tu df_universidad
df_total = df_total[['universidad', 'curso', 'carrera']]

# Mostrar el resultado final
print(df_total.head())

    id_curso id_universidad_x id_carrera  \
0          1                1          1   
1          1                1          1   
2          2                1          1   
3          2                1          1   
4          3                1          1   
..       ...              ...        ...   
373      187                3          1   
374      188                3          1   
375      188                3          1   
376      189                3          1   
377      189                3          1   

                                          curso ciclo id_universidad_y  \
0       Álgebra matricial y geometría analítica     1                1   
1       Álgebra matricial y geometría analítica     1                2   
2                        Comunicación académica     1                1   
3                        Comunicación académica     1                2   
4                        Fundamentos de cálculo     1                1   
..                         

# **otras u**

In [147]:
"""
extraer_mallas.py
------------------
Extrae cursos de mallas curriculares (PDF) hacia un DataFrame con columnas:
universidad, carrera, curso, ciclo

Funciona bien con mallas en formato TABLA/LISTA con texto seleccionable
(UNALM, UNI, ESAN, UPC, y similares). Las mallas que son infografías/imágenes
con diseño gráfico libre (p.ej. folletos de Universidad del Pacífico,
Universidad de Lima, Universidad de Piura, Científica del Sur) no tienen una
estructura de tabla real en el PDF, así que la extracción ahí será parcial o
nula -- estas se imprimen en el reporte final para que las revises a mano.

Requiere: pdfplumber  (pip install pdfplumber --break-system-packages)
"""

import re
import os
import pandas as pd
import pdfplumber

# ------------------------------------------------------------------
# 1) MAPEO DE ARCHIVOS -> UNIVERSIDAD / CARRERA
#    (ajusta las rutas si tus PDFs están en otra carpeta)
# ------------------------------------------------------------------

UPLOADS_DIR = "C:/Users/Mildred/Downloads/"

PDF_MAP = [
    # (archivo, universidad, carrera, grupo_de_parseo)
    ("Brochure-carrera-Ingenieria-Empresarial-admision-.2025.pdf",
     "Universidad del Pacífico", "Ingeniería Empresarial", "brochure"),

    ("00_DIG_BRO_Ing_Informacion_2026_Hoja-compressed.pdf",
     "Universidad del Pacífico", "Ingeniería de la Información", "brochure"),

    ("malla_curricular_ingenieria_de_sistemas_2026.pdf",
     "Universidad de Lima", "Ingeniería de Sistemas", "brochure"),

    ("malla-ING-2026.pdf",
     "Universidad de Piura", "Ingeniería", "brochure"),

    ("Plan de estudios Estadística Informática.pdf",
     "UNALM", "Estadística Informática", "codigo"),

    ("PLAN-ESTUDIOS-INGENIERIA-SISTEMAS-2018-v01_-_VERSION_01-pages-1-4.pdf",
     "UNI", "Ingeniería de Sistemas", "codigo"),

    ("PLAN_ESTUDIOS-_INGENIERIA-SOFTWARE_11-07-2021-CF-V1.pdf",
     "UNI", "Ingeniería de Software", "codigo"),

    ("PLAN-DE-ESTUDIOS-CIBER.pdf",
     "UNI", "Ingeniería de Ciberseguridad", "codigo"),

    ("ADMINISTRACIÓN Y CIENCIA DE DATOS PARA NEGOCIO PREGRADO FDM PR.pdf",
     "UPC", "Administración y Ciencia de Datos para Negocio", "codigo"),

    ("INGENIERIA DE SOFTWARE PREGRADO MW FDM PR.pdf",
     "UPC", "Ingeniería de Software", "codigo"),

    ("INGENIERIA DE SISTEMAS DE INFORMACION PREGRADO MW FDM PR.pdf",
     "UPC", "Ingeniería de Sistemas de Información", "codigo"),

    ("INGENIERÍA DE CIBERSEGURIDAD PREGRADO FDM PR.pdf",
     "UPC", "Ingeniería de Ciberseguridad", "codigo"),

    ("CIENCIAS DE LA COMPUTACION PREGRADO FDM PR.pdf",
     "UPC", "Ciencias de la Computación", "codigo"),

    ("Ingenieria-empresarial-y-de-sistemas-2026-cientifica.pdf",
     "Científica del Sur", "Ingeniería Empresarial y de Sistemas", "brochure"),

    ("Ingenieria-de-software-2026-cientifica.pdf",
     "Científica del Sur", "Ingeniería de Software", "brochure"),

    ("INGENIERIA-EN-INTELIGENCIA-ARTIFICIAL-Y-CIENCIA-DE-DATOS-CIENTIFICA.pdf",
     "Científica del Sur", "Ingeniería en Inteligencia Artificial y Ciencia de Datos", "brochure"),

    ("Malla_Ingenieria _de_tecnologias_de_Informacion.pdf",
     
     "ESAN", "Ingeniería de Tecnologías de Información y Sistemas", "esan"),

    ("Malla_Ingenieria_en_Inteligencia Artificial.pdf",
     "ESAN", "Ingeniería en Inteligencia Artificial", "esan"),
]

# ------------------------------------------------------------------
# 2) EXPRESIONES REGULARES
# ------------------------------------------------------------------

# Código de curso típico: letras+numeros (EP1055, BMA01, 1ADN0048, FB101, CBS01...)
CODE_RE = re.compile(r"^([0-9]?[A-ZÁÉÍÓÚÑ]{2,4}\d{2,5}|DEP|ELC|PP\d+|EG\d+)\s+(.*)$")

# Marcadores de ciclo en número: "CICLO 1", "CICLO: 01", "NIVEL 3", "▸▸ CICLO 7 22"
CYCLE_NUM_RE = re.compile(r"(?:CICLO|NIVEL)\s*[:\s]*0?(\d{1,2})\b", re.IGNORECASE)

# Marcadores de ciclo en texto ordinal: "PRIMER CICLO", "DECIMO CICLO", etc.
ORDINAL_MAP = {
    "CERO": 0, "PRIMER": 1, "PRIMERO": 1, "SEGUNDO": 2, "TERCER": 3, "TERCERO": 3,
    "CUARTO": 4, "QUINTO": 5, "SEXTO": 6, "SEPTIMO": 7, "SÉPTIMO": 7,
    "OCTAVO": 8, "NOVENO": 9, "DECIMO": 10, "DÉCIMO": 10,
}
ORDINAL_RE = re.compile(
    r"\b(" + "|".join(ORDINAL_MAP.keys()) + r")\b\s*CICLO", re.IGNORECASE
)

# Líneas que claramente no son cursos (encabezados, totales, leyendas)
SKIP_PATTERNS = re.compile(
    r"^(SUBTOTAL|TOTAL|LEYENDA|C[ÓO]DIGO|COD\b|ASIGNATURA|NOMBRE DEL CURSO|"
    r"CR[ÉE]DITOS|RESUMEN|CARÁCTER|FACULTAD|ESPECIALIDAD|PLAN DE ESTUDIOS|"
    r"CURSOS ELECTIVOS|HT\s+HP|TIPO\s+SIST|HST\s+HSP|CRD\s+REQUISITO)",
    re.IGNORECASE,
)

TRAILING_TYPE_TOKENS = {"O", "E", "A", "F", "G", "D", "B"}


def update_cycle(line, current_cycle):
    """Si la línea es un encabezado de ciclo, devuelve el nuevo número; si no, None."""
    m = CYCLE_NUM_RE.search(line)
    if m:
        return int(m.group(1))
    m = ORDINAL_RE.search(line)
    if m:
        return ORDINAL_MAP[m.group(1).upper()]
    return current_cycle


def _name_until_first_number(tokens):
    idx = len(tokens)
    for i, t in enumerate(tokens):
        if re.match(r"^\d+(\.\d+)?$", t):
            idx = i
            break
    return tokens[:idx]


def parse_line_codigo(line):
    """Para mallas con código de curso al inicio (UNALM, UNI, UPC)."""
    line = line.strip()
    if not line or SKIP_PATTERNS.match(line):
        return None

    m = CODE_RE.match(line)
    if m:
        rest = m.group(2)
    elif line.lower().startswith("electivo"):
        rest = line
    else:
        return None  # sin código reconocible -> no se interpreta como curso

    tokens = rest.split()
    name_tokens = _name_until_first_number(tokens)
    # quitar códigos de tipo/evaluación residuales al final (letras sueltas)
    while name_tokens and name_tokens[-1].upper() in TRAILING_TYPE_TOKENS and len(name_tokens[-1]) <= 2:
        name_tokens.pop()
    name = " ".join(name_tokens).strip(" .-")
    if len(name) < 3:
        return None
    return name


def parse_line_esan(line):
    """Para mallas ESAN: sin código, 'Nombre del curso  CREDITOS  TIPO  MODALIDAD  Requisitos'."""
    line = line.strip()
    if not line or SKIP_PATTERNS.match(line):
        return None
    tokens = line.split()
    name_tokens = _name_until_first_number(tokens)
    name = " ".join(name_tokens).strip(" .-")
    if len(name) < 3 or name.upper() == name:
        # nombres en mayúsculas sostenidas suelen ser encabezados, no cursos
        pass
    if len(name) < 3:
        return None
    return name


def extract_pdf_text(path):
    pages_text = []
    with pdfplumber.open(path) as pdf:
        for page in pdf.pages:
            txt = page.extract_text() or ""
            pages_text.append(txt)
    return pages_text


def process_pdf(path, universidad, carrera, grupo):
    rows = []
    if not os.path.exists(path):
        print(f"  [!] No encontrado: {path}")
        return rows

    pages_text = extract_pdf_text(path)
    current_cycle = None

    parser = parse_line_codigo if grupo in ("codigo",) else (
        parse_line_esan if grupo == "esan" else parse_line_codigo
    )

    for txt in pages_text:
        for raw_line in txt.split("\n"):
            line = raw_line.strip()
            if not line:
                continue

            new_cycle = update_cycle(line, current_cycle)
            if new_cycle != current_cycle:
                current_cycle = new_cycle
                # una línea de encabezado de ciclo normalmente no trae un curso útil
                # pero algunas mallas meten varios cursos en la misma línea de ciclo,
                # así que igual se intenta parsear más abajo si sobra texto.

            curso = parser(line)
            if curso:
                rows.append(
                    {
                        "universidad": universidad,
                        "carrera": carrera,
                        "curso": curso,
                        "ciclo": current_cycle,
                    }
                )
    return rows


def main():
    all_rows = []
    sin_estructura = []

    for filename, universidad, carrera, grupo in PDF_MAP:
        path = os.path.join(UPLOADS_DIR, filename)
        print(f"Procesando: {filename}  ->  {universidad} / {carrera}")
        rows = process_pdf(path, universidad, carrera, grupo)
        if grupo == "brochure" and len(rows) < 5:
            sin_estructura.append(filename)
        all_rows.extend(rows)

    df = pd.DataFrame(all_rows, columns=["universidad", "carrera", "curso", "ciclo"])

    # limpieza final: quitar duplicados exactos y filas con curso vacío
    df = df.drop_duplicates().reset_index(drop=True)
    df = df[df["curso"].str.len() > 2].reset_index(drop=True)

    out_path = "C:/Users/Mildred/Downloads/mallas_curriculares.csv"
    os.makedirs(os.path.dirname(out_path), exist_ok=True)
    df.to_csv(out_path, index=False, encoding="utf-8-sig")

    print("\n--- RESUMEN ---")
    print(df.groupby(["universidad", "carrera"]).size())
    print(f"\nTotal de filas extraídas: {len(df)}")
    print(f"Guardado en: {out_path}")


if __name__ == "__main__":
    main()

Procesando: Brochure-carrera-Ingenieria-Empresarial-admision-.2025.pdf  ->  Universidad del Pacífico / Ingeniería Empresarial
Procesando: 00_DIG_BRO_Ing_Informacion_2026_Hoja-compressed.pdf  ->  Universidad del Pacífico / Ingeniería de la Información
Procesando: malla_curricular_ingenieria_de_sistemas_2026.pdf  ->  Universidad de Lima / Ingeniería de Sistemas
Procesando: malla-ING-2026.pdf  ->  Universidad de Piura / Ingeniería
Procesando: Plan de estudios Estadística Informática.pdf  ->  UNALM / Estadística Informática
Procesando: PLAN-ESTUDIOS-INGENIERIA-SISTEMAS-2018-v01_-_VERSION_01-pages-1-4.pdf  ->  UNI / Ingeniería de Sistemas
Procesando: PLAN_ESTUDIOS-_INGENIERIA-SOFTWARE_11-07-2021-CF-V1.pdf  ->  UNI / Ingeniería de Software
Procesando: PLAN-DE-ESTUDIOS-CIBER.pdf  ->  UNI / Ingeniería de Ciberseguridad
Procesando: ADMINISTRACIÓN Y CIENCIA DE DATOS PARA NEGOCIO PREGRADO FDM PR.pdf  ->  UPC / Administración y Ciencia de Datos para Negocio
Procesando: INGENIERIA DE SOFTWARE PREGR

In [148]:
df = pd.read_csv('mallas_curriculares.csv')
df

,universidad,carrera,curso,ciclo
0,Universidad del Pacífico,Ingeniería de la Información,ELECTIVOS,0.0
1,Universidad de Lima,Ingeniería de Sistemas,ELECTIVO,1.0
2,UNALM,Estadística Informática,Administración General Obligatorio,1.0
3,UNALM,Estadística Informática,Análisis Matemático I General,1.0
4,UNALM,Estadística Informática,Curso de Deportes o Actividades Culturales Deportivo y/o Cultural,1.0
...,...,...,...,...
861,ESAN,Ingeniería en Inteligencia Artificial,Electivo de especialidad IX,10.0
862,ESAN,Ingeniería en Inteligencia Artificial,Electivo de especialidad X,10.0
863,ESAN,Ingeniería en Inteligencia Artificial,• CRÉDITOS: Número de crédito de la asignatura,10.0
864,ESAN,Ingeniería en Inteligencia Artificial,"• TIPO DE CURSO Tipo de curso, pude ser obligatorio (O) o electivo...",10.0


In [149]:
import re
import os
import pdfplumber
import pandas as pd

# ------------------------------------------------------------------
# 1) RUTAS DE ARCHIVOS
# ------------------------------------------------------------------

UPLOADS_DIR = "/mnt/user-data/uploads/"

PDF_MAP = [
    (
        "Brochure-carrera-Ingenieria-Empresarial-admision-_2025.pdf",
        "Universidad del Pacífico",
        "Ingeniería Empresarial",
    ),
    (
        "00_DIG_BRO_Ing_Informacion_2026_Hoja-compressed.pdf",
        "Universidad del Pacífico",
        "Ingeniería de la Información",
    ),
]

# ------------------------------------------------------------------
# 2) MALLAS HARDCODED (extraídas visualmente de los brochures UP)
#    Fuente: páginas de malla curricular en cada brochure
#    ciclo=None significa que el brochure no indica ciclo explícito
#    para ese curso (malla IE) o que es un curso sello/electivo libre.
# ------------------------------------------------------------------

# Ingeniería Empresarial — ciclos 00-10 (brochure admisión 2025)
MALLA_IE = {
    0:  ["Nivelación en Lenguaje", "Nivelación en Matemáticas", "Nivelación en Informática"],
    1:  ["Lenguaje I", "Matemáticas I", "Economía General I", "Introducción a la Ingeniería",
         "Design Thinking and Technological Innovation"],
    2:  ["Lenguaje II", "Economía General II", "Fundamentos de Contabilidad", "Matemáticas II"],
    3:  ["Investigación Académica", "Estadística I", "Contabilidad Financiera Intermedia",
         "Herramientas de Programación", "Métodos de Investigación Cuantitativa"],
    4:  ["Marketing Estratégico", "Estadística II", "Ingeniería de Procesos",
         "Arquitectura del Sistema de Información", "Formulación y Evaluación de Proyectos"],
    5:  ["Fundamentos de Finanzas", "Fundamentos de Analítica", "Ingeniería de Datos",
         "Investigación de Operaciones"],
    6:  ["Gestión del Capital Humano", "Física", "Procesos de Suministro", "Gestión de Proyectos"],
    7:  ["Estrategia", "Desarrollo de Soluciones Empresariales", "Procesos de Producción",
         "Metodologías Ágiles"],
    8:  ["Tecnología para el Desarrollo Sostenible", "Infraestructura Tecnológica",
         "Procesos de Distribución", "Transformación Digital y Gestión del Cambio"],
    9:  ["Gerencia de Ingeniería de Valor", "Emprendimiento Tecnológico",
         "Tecnología y Negocios Digitales"],
    10: ["Ética", "Proyección Social", "Trabajo Final de Ingeniería Empresarial"],
}

# Cursos Sello UP — obligatorios en todas las carreras (sin ciclo fijo en el brochure IE)
SELLO_UP_IE = [
    "Cursos de Pensamiento Crítico",
    "Cursos de Procesos Sociales",
    "Cursos de Introducción al Quehacer Científico",
    "Cursos de Ciencias Sociales",
    "Cursos de Desarrollo Personal",
]

# Ingeniería de la Información — ciclos 00-10 (brochure admisión 2027)
MALLA_II = {
    0:  ["Nivelación en Lenguaje", "Nivelación en Matemáticas", "Nivelación en Informática"],
    1:  ["Lenguaje I", "Matemáticas I", "Economía General I", "Introducción a la Ingeniería"],
    2:  ["Lenguaje II", "Matemáticas II", "Economía General II",
         "Programación para la Ciencia de Datos"],
    3:  ["Investigación Académica", "Estadística I", "Contabilidad Financiera Intermedia",
         "Herramientas de Programación", "Curso de Quehacer Científico",
         "Matemáticas Discretas para la Computación", "Arquitectura del Sistema de Información",
         "Curso de Desarrollo Personal"],
    4:  ["Marketing Estratégico", "Estadística II", "Ingeniería de Procesos",
         "Álgebra Lineal Aplicada", "Ingeniería de Datos"],
    5:  ["Fundamentos de Finanzas", "Fundamentos de Analítica", "Técnicas de Programación",
         "Machine Learning"],
    6:  ["Gestión del Capital Humano", "Física", "Data Mining",
         "Curso de Ciencias Sociales", "Curso de Procesos Sociales"],
    7:  ["Estrategia", "Inteligencia Computacional", "Desarrollo de Soluciones Empresariales",
         "Analítica de la Web", "Curso de Pensamiento Crítico"],
    8:  ["Tecnología para el Desarrollo Sostenible", "Deep Learning",
         "Computación de Alto Desempeño y Cloud Computing", "Infraestructura Tecnológica",
         "Curso de Procesos Sociales"],
    9:  ["Big Data Analytics", "Business Intelligence",
         "Trabajo Final de Ingeniería de la Información I",
         "Curso de Pensamiento Crítico", "Curso de Procesos Sociales"],
    10: ["Ética", "Proyección Social",
         "Trabajo Final de Ingeniería de la Información II"],
}

# Cursos Sello UP para Ingeniería de la Información
SELLO_UP_II = [
    "Curso de Quehacer Científico",
    "Curso de Ciencias Sociales",
    "Curso de Pensamiento Crítico",
    "Curso de Procesos Sociales",
    "Curso de Desarrollo Personal",
]

# ------------------------------------------------------------------
# 3) CONSTRUCCIÓN DEL DATAFRAME DESDE LAS MALLAS HARDCODED
# ------------------------------------------------------------------

def build_rows_from_hardcoded():
    rows = []

    for ciclo, cursos in MALLA_IE.items():
        for curso in cursos:
            rows.append({
                "universidad": "Universidad del Pacífico",
                "carrera": "Ingeniería Empresarial",
                "curso": curso,
                "ciclo": ciclo,
            })
    # Sello UP sin ciclo fijo
    for curso in SELLO_UP_IE:
        rows.append({
            "universidad": "Universidad del Pacífico",
            "carrera": "Ingeniería Empresarial",
            "curso": curso,
            "ciclo": None,
        })

    for ciclo, cursos in MALLA_II.items():
        for curso in cursos:
            rows.append({
                "universidad": "Universidad del Pacífico",
                "carrera": "Ingeniería de la Información",
                "curso": curso,
                "ciclo": ciclo,
            })
    for curso in SELLO_UP_II:
        # solo añadir si no está ya en la malla con ciclo
        if not any(r["carrera"] == "Ingeniería de la Información"
                   and r["curso"] == curso for r in rows):
            rows.append({
                "universidad": "Universidad del Pacífico",
                "carrera": "Ingeniería de la Información",
                "curso": curso,
                "ciclo": None,
            })

    return rows

# ------------------------------------------------------------------
# 4) EXTRACCIÓN ADICIONAL DESDE PDF (complementa lo hardcoded)
#    Útil si en el futuro los PDFs tienen texto más seleccionable.
# ------------------------------------------------------------------

CYCLE_NUM_RE = re.compile(r"CICLO\s*0?(\d{1,2})\b", re.IGNORECASE)

SKIP_RE = re.compile(
    r"^(INGENIERÍA|UNIVERSIDAD|ADMISIÓN|MALLA|CURSOS|PLAN DE ESTUDIOS|"
    r"CICLO\s*\d|UP\b|Hasta|Malla|Fuente|Bachiller|Maestría|Doble|"
    r"Sé parte|Vive|Diseña|Conecta|Participa|Aplica|Forma|Lleva|"
    r"Desarrolla|Natural Language|Optimización|Inteligencia Artificial Gen)",
    re.IGNORECASE,
)

def extract_extra_from_pdf(path, universidad, carrera):
    """Intenta extraer cursos adicionales del PDF como complemento."""
    rows = []
    if not os.path.exists(path):
        print(f"  [!] Archivo no encontrado: {path}")
        return rows

    current_cycle = None
    try:
        with pdfplumber.open(path) as pdf:
            for page in pdf.pages:
                text = page.extract_text() or ""
                for raw_line in text.split("\n"):
                    line = raw_line.strip()
                    if not line:
                        continue
                    m = CYCLE_NUM_RE.search(line)
                    if m:
                        current_cycle = int(m.group(1))
                        continue
                    if SKIP_RE.match(line):
                        continue
                    # Heurística: línea de curso tiene entre 5 y 80 chars,
                    # no es todo mayúsculas y tiene al menos 2 palabras.
                    words = line.split()
                    if (5 <= len(line) <= 80
                            and len(words) >= 2
                            and not line.isupper()):
                        rows.append({
                            "universidad": universidad,
                            "carrera": carrera,
                            "curso": line,
                            "ciclo": current_cycle,
                        })
    except Exception as e:
        print(f"  [!] Error leyendo {path}: {e}")
    return rows


# ------------------------------------------------------------------
# 5) MAIN
# ------------------------------------------------------------------

def main():
    all_rows = []

    # Parte A: malla hardcoded (garantizada)
    print("Construyendo mallas desde datos extraídos visualmente...")
    all_rows.extend(build_rows_from_hardcoded())
    print(f"  → {len(all_rows)} filas desde mallas hardcoded")

    # Parte B: complementar con lo que se pueda leer del PDF
    print("\nComplementando con extracción directa de PDFs...")
    for filename, universidad, carrera in PDF_MAP:
        path = os.path.join(UPLOADS_DIR, filename)
        print(f"  Leyendo: {filename}")
        extra = extract_extra_from_pdf(path, universidad, carrera)
        print(f"    → {len(extra)} líneas candidatas desde PDF")
        all_rows.extend(extra)

    # Construcción del DataFrame
    df = pd.DataFrame(all_rows, columns=["universidad", "carrera", "curso", "ciclo"])

    # Limpiar: eliminar duplicados exactos y cursos muy cortos
    df = df.drop_duplicates(subset=["carrera", "curso"]).reset_index(drop=True)
    df = df[df["curso"].str.len() >= 5].reset_index(drop=True)

    # Ordenar por carrera y ciclo
    df = df.sort_values(["carrera", "ciclo"], na_position="last").reset_index(drop=True)

    # Guardar CSV
    out_path = "/mnt/user-data/outputs/mallas_UP.csv"
    df.to_csv(out_path, index=False, encoding="utf-8-sig")

    print("\n--- RESUMEN ---")
    print(df.groupby(["universidad", "carrera"]).size().to_string())
    print(f"\nTotal de filas: {len(df)}")
    print(f"Guardado en: {out_path}")

    return df


if __name__ == "__main__":
    df = main()
    print("\nPrimeras filas:")
    print(df.head(20).to_string(index=False))

Construyendo mallas desde datos extraídos visualmente...
  → 100 filas desde mallas hardcoded

Complementando con extracción directa de PDFs...
  Leyendo: Brochure-carrera-Ingenieria-Empresarial-admision-_2025.pdf
  [!] Archivo no encontrado: /mnt/user-data/uploads/Brochure-carrera-Ingenieria-Empresarial-admision-_2025.pdf
    → 0 líneas candidatas desde PDF
  Leyendo: 00_DIG_BRO_Ing_Informacion_2026_Hoja-compressed.pdf
  [!] Archivo no encontrado: /mnt/user-data/uploads/00_DIG_BRO_Ing_Informacion_2026_Hoja-compressed.pdf
    → 0 líneas candidatas desde PDF

--- RESUMEN ---
universidad               carrera                     
Universidad del Pacífico  Ingeniería Empresarial          49
                          Ingeniería de la Información    48

Total de filas: 97
Guardado en: /mnt/user-data/outputs/mallas_UP.csv

Primeras filas:
             universidad                carrera                                        curso  ciclo
Universidad del Pacífico Ingeniería Empresarial        

In [150]:
df

,universidad,carrera,curso,ciclo
0,Universidad del Pacífico,Ingeniería Empresarial,Nivelación en Lenguaje,0.0
1,Universidad del Pacífico,Ingeniería Empresarial,Nivelación en Matemáticas,0.0
2,Universidad del Pacífico,Ingeniería Empresarial,Nivelación en Informática,0.0
3,Universidad del Pacífico,Ingeniería Empresarial,Lenguaje I,1.0
4,Universidad del Pacífico,Ingeniería Empresarial,Matemáticas I,1.0
5,Universidad del Pacífico,Ingeniería Empresarial,Economía General I,1.0
6,Universidad del Pacífico,Ingeniería Empresarial,Introducción a la Ingeniería,1.0
7,Universidad del Pacífico,Ingeniería Empresarial,Design Thinking and Technological Innovation,1.0
8,Universidad del Pacífico,Ingeniería Empresarial,Lenguaje II,2.0
9,Universidad del Pacífico,Ingeniería Empresarial,Economía General II,2.0


In [151]:
import re
import os
import pdfplumber
import pandas as pd

# ------------------------------------------------------------------
# 1) MAPEO DE ARCHIVOS UPC
# ------------------------------------------------------------------

UPLOADS_DIR = "C:/Users/Mildred/Downloads/"

PDF_MAP_UPC = [
    (
        "ADMINISTRACIÓN Y CIENCIA DE DATOS PARA NEGOCIO PREGRADO FDM PR.pdf",
        "UPC",
        "Administración y Ciencia de Datos para Negocio",
    ),
    (
        "INGENIERIA DE SOFTWARE PREGRADO MW FDM PR.pdf",
        "UPC",
        "Ingeniería de Software",
    ),
    (
        "INGENIERIA DE SISTEMAS DE INFORMACION PREGRADO MW FDM PR.pdf",
        "UPC",
        "Ingeniería de Sistemas de Información",
    ),
    (
        "INGENIERÍA DE CIBERSEGURIDAD PREGRADO FDM PR.pdf",
        "UPC",
        "Ingeniería de Ciberseguridad",
    ),
    (
        "CIENCIAS DE LA COMPUTACION PREGRADO FDM PR.pdf",
        "UPC",
        "Ciencias de la Computación",
    ),
]

# ------------------------------------------------------------------
# 2) EXPRESIONES REGULARES
# ------------------------------------------------------------------

# Encabezado de ciclo: "1 ▸▸ CICLO 1 18"  o  "▸▸ CICLO 9 21"
CYCLE_RE = re.compile(r"CICLO\s+(\d{1,2})", re.IGNORECASE)

# Código de curso UPC: comienza con dígito opcional + 2-4 letras + 3-5 dígitos
#   Ej: 1AHU0712  1ADN0048  1AMA0711  SI769  SI792
CODE_RE = re.compile(
    r"^((?:[0-9]?[A-ZÁÉÍÓÚÑ]{2,4}\d{3,5})|(?:[A-Z]{2,4}\d{3,4}))\s+(.+)"
)

# Tokens basura al final del nombre (números, créditos, "General", "Carrera", etc.)
# Cortamos el nombre al primer token que sea número puro, "General" o "Carrera"
STOP_WORDS = {"General", "Carrera", "Electivo"}

def clean_name(rest: str) -> str:
    """Extrae el nombre del curso antes de los datos numéricos / tipo."""
    tokens = rest.split()
    name_tokens = []
    for t in tokens:
        # Parar en número puro (horas, créditos) o palabras clave de tipo
        if re.match(r"^\d+(\.\d+)?$", t):
            break
        if t in STOP_WORDS:
            break
        name_tokens.append(t)
    name = " ".join(name_tokens).strip(" .-,")
    return name

# ------------------------------------------------------------------
# 3) EXTRACCIÓN DESDE PDF
# ------------------------------------------------------------------

def extract_upc_pdf(path: str, universidad: str, carrera: str) -> list[dict]:
    rows = []
    if not os.path.exists(path):
        print(f"  [!] No encontrado: {path}")
        return rows

    current_cycle = None

    with pdfplumber.open(path) as pdf:
        for page in pdf.pages:
            text = page.extract_text() or ""
            for raw_line in text.split("\n"):
                line = raw_line.strip()
                if not line:
                    continue

                # Detectar encabezado de ciclo
                m_cycle = CYCLE_RE.search(line)
                if m_cycle:
                    current_cycle = int(m_cycle.group(1))
                    continue  # la línea de ciclo no contiene un curso

                # Detectar línea de curso por código
                m_course = CODE_RE.match(line)
                if m_course:
                    nombre = clean_name(m_course.group(2))
                    if len(nombre) >= 4:
                        rows.append({
                            "universidad": universidad,
                            "carrera": carrera,
                            "curso": nombre,
                            "ciclo": current_cycle,
                        })

    return rows

# ------------------------------------------------------------------
# 4) MAIN
# ------------------------------------------------------------------

def main():
    # ── A) Cargar CSV previo (UP) ──────────────────────────────────
    prev_path = "C:/Users/Mildred/Downloads/mallas_UP_UPC.csv"
    if os.path.exists(prev_path):
        df_prev = pd.read_csv(prev_path, encoding="utf-8-sig")
        print(f"CSV previo cargado: {len(df_prev)} filas  ({prev_path})")
    else:
        df_prev = pd.DataFrame(columns=["universidad", "carrera", "curso", "ciclo"])
        print("  [!] CSV previo no encontrado, se creará uno nuevo.")

    # ── B) Extraer mallas UPC ──────────────────────────────────────
    all_rows = []
    for filename, universidad, carrera in PDF_MAP_UPC:
        path = os.path.join(UPLOADS_DIR, filename)
        print(f"Procesando: {carrera}")
        rows = extract_upc_pdf(path, universidad, carrera)
        print(f"  → {len(rows)} cursos extraídos")
        all_rows.extend(rows)

    df_upc = pd.DataFrame(all_rows, columns=["universidad", "carrera", "curso", "ciclo"])
    df_upc = df_upc.drop_duplicates(subset=["carrera", "curso"]).reset_index(drop=True)
    df_upc = df_upc[df_upc["curso"].str.len() >= 4].reset_index(drop=True)
    df_upc = df_upc.sort_values(["carrera", "ciclo"]).reset_index(drop=True)

    # ── C) Concatenar UP + UPC ─────────────────────────────────────
    df_total = pd.concat([df, df_upc], ignore_index=True)

    out_path = "C:/Users/Mildred/Downloads/mallas_new.csv"
    df_total.to_csv(out_path, index=False, encoding="utf-8-sig")

    # ── D) Resumen ─────────────────────────────────────────────────
    print("\n--- RESUMEN FINAL ---")
    resumen = df_total.groupby(["universidad", "carrera"]).size().rename("cursos")
    print(resumen.to_string())
    print(f"\nTotal filas: {len(df_total)}")
    print(f"Guardado en: {out_path}")

    return df_total


if __name__ == "__main__":
    df = main()

CSV previo cargado: 313 filas  (C:/Users/Mildred/Downloads/mallas_UP_UPC.csv)
Procesando: Administración y Ciencia de Datos para Negocio
  → 50 cursos extraídos
Procesando: Ingeniería de Software
  → 42 cursos extraídos
Procesando: Ingeniería de Sistemas de Información
  → 41 cursos extraídos
Procesando: Ingeniería de Ciberseguridad
  → 41 cursos extraídos
Procesando: Ciencias de la Computación
  → 42 cursos extraídos

--- RESUMEN FINAL ---
universidad               carrera                                       
UPC                       Administración y Ciencia de Datos para Negocio    50
                          Ciencias de la Computación                        42
                          Ingeniería de Ciberseguridad                      41
                          Ingeniería de Sistemas de Información             41
                          Ingeniería de Software                            42
Universidad del Pacífico  Ingeniería Empresarial                            49
        

In [152]:
df

,universidad,carrera,curso,ciclo
0,Universidad del Pacífico,Ingeniería Empresarial,Nivelación en Lenguaje,0.0
1,Universidad del Pacífico,Ingeniería Empresarial,Nivelación en Matemáticas,0.0
2,Universidad del Pacífico,Ingeniería Empresarial,Nivelación en Informática,0.0
3,Universidad del Pacífico,Ingeniería Empresarial,Lenguaje I,1.0
4,Universidad del Pacífico,Ingeniería Empresarial,Matemáticas I,1.0
...,...,...,...,...
308,UPC,Ingeniería de Software,Calidad y Mejora de Procesos de Software,9.0
309,UPC,Ingeniería de Software,Desarrollo Agile de Productos de Software,9.0
310,UPC,Ingeniería de Software,Fundamentos de Investigación Académica y aprobación por el Directo...,9.0
311,UPC,Ingeniería de Software,Trabajo de Investigación I,9.0


In [153]:
import os
import pandas as pd

# ------------------------------------------------------------------
# 1) MALLAS HARDCODED — Científica del Sur
#    Fuente: brochures PDF admisión 2026
#    Los ciclos están indicados con números del 1 al 10 en los brochures.
#    Los cursos que aparecen sin ciclo explícito (electivos genéricos)
#    se registran con ciclo=None.
# ------------------------------------------------------------------

UNIVERSIDAD = "Científica del Sur"

# ── Ingeniería Empresarial y de Sistemas ──────────────────────────
MALLA_IES = {
    1: [
        "Matemática I",
        "Álgebra",
        "Desempeño Universitario",
        "Lengua y Comunicación",
        "Algoritmos y Programación",
        "Fundamentos de Ingeniería Empresarial y de Sistemas",
    ],
    2: [
        "Arquitectura de Tecnologías de Información",
        "Diseño Lógico Digital y Arquitectura de Computadoras",
        "Information Technology Management and Systems Audits",
        "Redes y Comunicaciones",
        "Innovación y Creatividad",
        "Investigación de Proyectos",
    ],
    3: [
        "Gestión de Calidad",
        "Formulación y Evaluación de Proyecto IES",
        "Lenguaje de Programación I",
        "Realidad Nacional",
        "Estadística General",
        "Estructura de la Información",
    ],
    4: [
        "Educación Ambiental",
        "Sistemas Operativos",
        "Física I para Ingeniería",
        "Inglés",
        "Lenguaje de Programación II",
        "Análisis, Diseño y Desarrollo de Sistemas",
    ],
    5: [
        "Matemática II",
        "Fundamentos de Logística y Operaciones",
        "Lengua y Comunicación II",
        "Química para Ingeniería",
        "Administración y Organizaciones",
        "Contabilidad Gerencial",
    ],
    6: [
        "Finanzas",
        "Dirección de Empresas",
        "Estrategia, Competitividad y Riesgo",
        "Sistemas y Procesos Industriales",
        "Ética y Deontología Profesional",
        "Planificación Estratégica",
    ],
    7: [
        "Ingeniería de Requerimientos",
        "Física II para Ingeniería",
        "Arquitectura Empresarial",
        "IA aplicada",
        "Analítica y Big Data",
        "Inteligencia Artificial Basada en el Conocimiento",
    ],
    8: [
        "Visualización de Datos",
        "Bases de Datos y Big Data",
        "Introducción a Data Science",
        "Matemática y Estadística para Ingenieros",
        "Introducción a la Investigación",
        "Seminario de Investigación",
    ],
    9: [
        "Trabajo de Investigación",
        "Investigación de Operaciones",
        "Tecnologías Emergentes",
        "Administración de Proyectos TI",
        "Management 3.0",
        "Marketing Digital",
    ],
    10: [
        "Machine Learning",
        "Ingeniería y Gestión de Procesos de Negocio",
    ],
}

ELECTIVOS_IES = ["Electivo"] * 2   # 2 electivos sin ciclo fijo

# ── Ingeniería de Software ─────────────────────────────────────────
MALLA_IS = {
    1: [
        "Matemática I",
        "Desempeño Universitario",
        "Lengua y Comunicación",
        "MUN Skills",
        "Introducción a la Ingeniería de Software",
        "Fundamentos Científicos",
    ],
    2: [
        "Matemática II",
        "Fundamentos de IA y Programación",
        "Realidad Nacional",
        "Educación Ambiental",
        "Empresa, Sociedad y Gobierno",
        "Física Aplicada",
    ],
    3: [
        "Matemática III",
        "Introducción a la Investigación",
        "Estadística General",
        "Algoritmos y Estructura de Datos",
        "Fundamentos de Base de Datos",
        "Introducción a Data Science",
    ],
    4: [
        "Applied Innovation Project",
        "Metodologías Ágiles y Gestión Lean",
        "Lenguaje de Programación II",
        "Sistemas Avanzados de Base de Datos",
        "Análisis y Diseño de Software",
        "CTO and Product Leadership",
    ],
    5: [
        "Redes y Comunicaciones",
        "Arquitectura de Computadoras",
        "Construcción de Software y Frameworks",
        "Blockchain",
        "Machine Learning I",
        "Desarrollo Full-Stack y Arquitectura Web",
    ],
    6: [
        "Verificación y Validación de Software",
        "Control de Calidad de Software",
        "Taller de Ingeniería de Software I",
        "Machine Learning II",
        "Consultoría en Desarrollo de Software",
        "Innovation & Entrepreneurship Lab",
    ],
    7: [
        "Fundamentos de Transformación Digital",
        "AI & Data Analytics para Ingenieros",
        "Sistemas Operativos",
        "Lenguaje de Programación I",
    ],
    8: [
        "Capstone: Proyecto Integrador de Software Engineering",
        "AI Power Software Development",
        "Tecnologías Emergentes",
        "Taller de Ingeniería de Software II",
        "DevOps Avanzado",
        "Seminario de Investigación",
    ],
    9: [
        "Trabajo de Investigación",
    ],
    10: [],  # solo electivos en ciclo 10
}

ELECTIVOS_IS = ["Electivo"] * 5   # electivos libres (5 en total según brochure)

# ── Ingeniería en Inteligencia Artificial y Ciencia de Datos ──────
MALLA_IACD = {
    1: [
        "Matemática I",
        "Desempeño Universitario",
        "Lengua y Comunicación",
        "MUN Skills",
        "Introducción a la Ingeniería en Inteligencia Artificial y Ciencia de Datos",
        "Fundamentos Científicos",
    ],
    2: [
        "Matemática II",
        "Fundamentos de IA y Programación",
        "Realidad Nacional",
        "Educación Ambiental",
        "Empresa, Sociedad y Gobierno",
        "Física Aplicada",
    ],
    3: [
        "Matemática III",
        "Introducción a la Investigación",
        "Algoritmos y Estructura de Datos",
        "Estadística General",
        "Fundamentos de Base de Datos",
        "Introducción a Data Science",
    ],
    4: [
        "Applied Innovation Project",
        "Metodologías Ágiles y Gestión Lean",
        "Business Analytics",
        "Datasets e Investigación en IA",
        "Machine Learning y Modelado Estadístico",
        "Laboratorio de Agentes IA y Sistemas Autónomos",
    ],
    5: [
        "Deep Learning y Redes Neuronales",
        "Sistemas Avanzados de Bases de Datos",
        "Storytelling y Visualización de Datos",
        "Estrategia de Gobierno de Datos",
        "Calidad de Datos",
        "Procesamiento de Lenguaje Natural y Visión Computacional",
    ],
    6: [
        "Big Data y Computación en la Nube",
        "Ética e IA Responsable",
        "Analítica Empresarial Avanzada y Business Intelligence",
        "Innovation & Entrepreneurship Lab",
        "Programación en R para Analytics",
        "Fundamentos de Transformación Digital",
    ],
    7: [
        "SQL para Analytics",
        "Programación Python para IA y Aplicaciones de Datos",
        "Ingeniería de Datos y Arquitectura de Data Lakes",
        "Seminario de Investigación",
    ],
    8: [
        "Capstone: Proyecto Integrador de IA y Ciencia de Datos",
        "Sistemas de IA y MLOps",
        "Humanidades Digitales y Sociedad (Pensamiento Crítico)",
    ],
    9: [
        "Trabajo de Investigación",
    ],
    10: [],  # solo electivos en ciclo 10
}

ELECTIVOS_IACD = ["Electivo"] * 5  # 5 electivos según brochure

# ------------------------------------------------------------------
# 2) CONSTRUCCIÓN DEL DATAFRAME
# ------------------------------------------------------------------

def build_rows(malla: dict, electivos: list, carrera: str) -> list[dict]:
    rows = []
    for ciclo, cursos in malla.items():
        for curso in cursos:
            rows.append({
                "universidad": UNIVERSIDAD,
                "carrera": carrera,
                "curso": curso,
                "ciclo": ciclo,
            })
    for i, curso in enumerate(electivos, start=1):
        rows.append({
            "universidad": UNIVERSIDAD,
            "carrera": carrera,
            "curso": f"Electivo {i}",
            "ciclo": None,
        })
    return rows



    # ── A) Construir DataFrame Científica ──────────────────────────
all_rows = []
carreras = [
    ("Ingeniería Empresarial y de Sistemas", MALLA_IES, ELECTIVOS_IES),
    ("Ingeniería de Software",               MALLA_IS,  ELECTIVOS_IS),
    ("Ingeniería en Inteligencia Artificial y Ciencia de Datos", MALLA_IACD, ELECTIVOS_IACD),
    ]

for carrera, malla, electivos in carreras:
    rows = build_rows(malla, electivos, carrera)
    print(f"  {carrera}: {len(rows)} filas")
    all_rows.extend(rows)

df_cientifica = pd.DataFrame(all_rows, columns=["universidad", "carrera", "curso", "ciclo"])
df_cientifica = (df_cientifica
                    .drop_duplicates(subset=["carrera", "curso"])
                    .reset_index(drop=True))

    # ── B) Cargar CSV previo (UP + UPC) ───────────────────────────
    # ── C) Concatenar y guardar ───────────────────────────────────
df = pd.concat([df, df_cientifica], ignore_index=True)


    # ── D) Resumen ────────────────────────────────────────────────
print("\n--- RESUMEN FINAL ---")
resumen = df.groupby(["universidad", "carrera"]).size().rename("cursos")
print(resumen.to_string())
print(f"\nTotal filas: {len(df)}")
print(f"Guardado en: {out_path}")



  Ingeniería Empresarial y de Sistemas: 58 filas
  Ingeniería de Software: 52 filas
  Ingeniería en Inteligencia Artificial y Ciencia de Datos: 49 filas

--- RESUMEN FINAL ---
universidad               carrera                                                 
Científica del Sur        Ingeniería Empresarial y de Sistemas                        58
                          Ingeniería de Software                                      52
                          Ingeniería en Inteligencia Artificial y Ciencia de Datos    49
UPC                       Administración y Ciencia de Datos para Negocio              50
                          Ciencias de la Computación                                  42
                          Ingeniería de Ciberseguridad                                41
                          Ingeniería de Sistemas de Información                       41
                          Ingeniería de Software                                      42
Universidad del Pacífico  Ing

NameError: name 'out_path' is not defined

In [154]:
df

,universidad,carrera,curso,ciclo
0,Universidad del Pacífico,Ingeniería Empresarial,Nivelación en Lenguaje,0.0
1,Universidad del Pacífico,Ingeniería Empresarial,Nivelación en Matemáticas,0.0
2,Universidad del Pacífico,Ingeniería Empresarial,Nivelación en Informática,0.0
3,Universidad del Pacífico,Ingeniería Empresarial,Lenguaje I,1.0
4,Universidad del Pacífico,Ingeniería Empresarial,Matemáticas I,1.0
...,...,...,...,...
467,Científica del Sur,Ingeniería en Inteligencia Artificial y Ciencia de Datos,Electivo 1,NaN
468,Científica del Sur,Ingeniería en Inteligencia Artificial y Ciencia de Datos,Electivo 2,NaN
469,Científica del Sur,Ingeniería en Inteligencia Artificial y Ciencia de Datos,Electivo 3,NaN
470,Científica del Sur,Ingeniería en Inteligencia Artificial y Ciencia de Datos,Electivo 4,NaN


In [155]:
import re
import os
import pdfplumber
import pandas as pd

UPLOADS_DIR = "C:/Users/Mildred/Downloads/"

PDF_MAP_ESAN = [
    (
        "Malla_Ingenieria _de_tecnologias_de_Informacion.pdf",
        "ESAN",
        "Ingeniería de Tecnologías de Información y Sistemas",
    ),
    (
        "Malla_Ingenieria_en_Inteligencia Artificial.pdf",
        "ESAN",
        "Ingeniería en Inteligencia Artificial",
    ),
]

ORDINAL_MAP = {
    "CERO": 0, "PRIMER": 1, "PRIMERO": 1,
    "SEGUNDO": 2, "TERCER": 3, "TERCERO": 3,
    "CUARTO": 4, "QUINTO": 5, "SEXTO": 6,
    "SÉPTIMO": 7, "SEPTIMO": 7, "OCTAVO": 8,
    "NOVENO": 9, "DÉCIMO": 10, "DECIMO": 10,
}
CYCLE_RE = re.compile(
    r"\b(" + "|".join(ORDINAL_MAP.keys()) + r")\s+CICLO\b",
    re.IGNORECASE,
)

SKIP_NAMES = {"ASIGNATURA", "SUBTOTAL", "TOTAL", "LEYENDA"}
ELECTIVO_RE = re.compile(r"^Electivo", re.IGNORECASE)


def parse_esan_pdf(path: str, universidad: str, carrera: str) -> list[dict]:
    rows = []
    current_cycle = None

    with pdfplumber.open(path) as pdf:
        for page in pdf.pages:
            for table in page.extract_tables():
                for row in table:
                    if not row or not row[0]:
                        continue
                    cell0 = str(row[0]).strip()

                    # Detectar encabezado de ciclo (fila con None en demás cols)
                    m = CYCLE_RE.search(cell0.upper())
                    if m:
                        current_cycle = ORDINAL_MAP[m.group(1).upper()]
                        continue

                    # Saltar encabezados de columna y totales
                    if cell0.upper() in SKIP_NAMES:
                        continue
                    if cell0.upper().startswith("SUBTOTAL") or cell0.upper().startswith("TOTAL"):
                        continue
                    # Saltar notas al pie
                    if cell0.startswith("*") or cell0.startswith("•"):
                        continue

                    nombre = cell0.strip(" .,*")
                    if len(nombre) < 4:
                        continue

                    # Normalizar electivos
                    if ELECTIVO_RE.match(nombre):
                        nombre = "Electivo"

                    rows.append({
                        "universidad": universidad,
                        "carrera":     carrera,
                        "curso":       nombre,
                        "ciclo":       current_cycle,
                    })

    # Deduplicar por (carrera, curso, ciclo)
    seen, unique = set(), []
    for r in rows:
        key = (r["carrera"], r["curso"], r["ciclo"])
        if key not in seen:
            seen.add(key)
            unique.append(r)

    return unique


# ------------------------------------------------------------------
# Construir df_esan y concatenar al df existente
# ------------------------------------------------------------------

all_rows = []
for filename, universidad, carrera in PDF_MAP_ESAN:
    path = os.path.join(UPLOADS_DIR, filename)
    rows = parse_esan_pdf(path, universidad, carrera)
    print(f"  {carrera}: {len(rows)} cursos")
    all_rows.extend(rows)

df_esan = pd.DataFrame(all_rows, columns=["universidad", "carrera", "curso", "ciclo"])

df = pd.concat([df, df_esan], ignore_index=True)

print(f"\n--- RESUMEN df actualizado ---")
print(df.groupby(["universidad", "carrera"]).size().rename("cursos").to_string())
print(f"\nTotal filas: {len(df)}")

  Ingeniería de Tecnologías de Información y Sistemas: 69 cursos
  Ingeniería en Inteligencia Artificial: 63 cursos

--- RESUMEN df actualizado ---
universidad               carrera                                                 
Científica del Sur        Ingeniería Empresarial y de Sistemas                        58
                          Ingeniería de Software                                      52
                          Ingeniería en Inteligencia Artificial y Ciencia de Datos    49
ESAN                      Ingeniería de Tecnologías de Información y Sistemas         69
                          Ingeniería en Inteligencia Artificial                       63
UPC                       Administración y Ciencia de Datos para Negocio              50
                          Ciencias de la Computación                                  42
                          Ingeniería de Ciberseguridad                                41
                          Ingeniería de Sistemas de Infor

In [156]:
df

,universidad,carrera,curso,ciclo
0,Universidad del Pacífico,Ingeniería Empresarial,Nivelación en Lenguaje,0.0
1,Universidad del Pacífico,Ingeniería Empresarial,Nivelación en Matemáticas,0.0
2,Universidad del Pacífico,Ingeniería Empresarial,Nivelación en Informática,0.0
3,Universidad del Pacífico,Ingeniería Empresarial,Lenguaje I,1.0
4,Universidad del Pacífico,Ingeniería Empresarial,Matemáticas I,1.0
...,...,...,...,...
599,ESAN,Ingeniería en Inteligencia Artificial,Tesis I,9.0
600,ESAN,Ingeniería en Inteligencia Artificial,Electivo,9.0
601,ESAN,Ingeniería en Inteligencia Artificial,Taller: Desarrollo de competencias profesionales V,10.0
602,ESAN,Ingeniería en Inteligencia Artificial,Tesis II,10.0


In [157]:
import re
import subprocess
import pandas as pd
from pathlib import Path


# ─────────────────────────────────────────────
# 1. CONFIGURACIÓN DE ARCHIVOS
# ─────────────────────────────────────────────

ARCHIVOS = [
    {
        "path": "Plan de estudios Estadística Informática.pdf",
        "tipo": "unalm",
    },
    {
        "path": "PLAN-ESTUDIOS-INGENIERIA-SISTEMAS-2018-v01_-_VERSION_01-pages-1-4.pdf",
        "tipo": "uni_sistemas",
    },
    {
        "path": "PLAN_ESTUDIOS-_INGENIERIA-SOFTWARE_11-07-2021-CF-V1.pdf",
        "tipo": "uni_software",
    },
]

PDF_DIR = Path("C:/Users/Mildred/Downloads/")   # ← ajusta si corres localmente


# ─────────────────────────────────────────────
# 2. EXTRACCIÓN DE TEXTO CON LAYOUT
# ─────────────────────────────────────────────
import pdfplumber

# Reemplaza tu función extraer_texto_pdf por esta:
def extraer_texto_pdf(ruta_pdf: Path) -> str:
    """
    Usa pdfplumber con layout=True para conservar la alineación 
    de columnas, imitando el comportamiento de pdftotext -layout.
    """
    texto_completo = []
    try:
        with pdfplumber.open(ruta_pdf) as pdf:
            for page in pdf.pages:
                # layout=True mantiene los espacios visuales entre columnas
                texto = page.extract_text(layout=True)
                if texto:
                    texto_completo.append(texto)
        return "\n".join(texto_completo)
    except Exception as e:
        raise RuntimeError(f"Error leyendo {ruta_pdf} con pdfplumber: {e}")
# 3. FUNCIONES DE LIMPIEZA
# ─────────────────────────────────────────────

# Patrón de código de curso: letras + guión opcional + números
PAT_CODIGO = r"[A-Z]{1,3}[-]?\d{3,5}"

# Prerequisito: código(s) o NINGUNO, separados por espacios / comas / barras
PAT_PREREQ = rf"(?:(?:{PAT_CODIGO}|NINGUNO|PP\d{{4}})[\s,/]*)"


def limpiar_nombre_curso(nombre: str) -> str:
    """
    Elimina del nombre del curso:
    1. Códigos de prerequisitos que hayan quedado pegados al final.
    2. Palabras clave de tipo de curso (Obligatorio, General…).
    3. Secuencias de números sueltos (créditos / horas).
    4. Espacios internos excesivos.
    """
    # Eliminar prereqs pegados al final: secuencias de códigos o NINGUNO
    nombre = re.sub(
        rf"\s+(?:{PAT_PREREQ})+$",
        "", nombre, flags=re.IGNORECASE
    )
    # Eliminar palabras de tipo de curso
    nombre = re.sub(
        r"\s+(Obligatorio|General|Electivo[\w\s]*|Deportivo[\w\s]*)$",
        "", nombre, flags=re.IGNORECASE
    )
    # Eliminar residuos numéricos al final
    nombre = re.sub(r"(\s+\d+){2,}\s*$", "", nombre)
    # Colapsar espacios internos
    nombre = re.sub(r"\s+", " ", nombre)
    return nombre.strip()


CICLO_MAP = {
    "PRIMER": "01", "SEGUNDO": "02", "TERCER": "03", "CUARTO": "04",
    "QUINTO": "05", "SEXTO": "06", "SEPTIMO": "07", "SÉPTIMO": "07",
    "OCTAVO": "08", "NOVENO": "09", "DECIMO": "10", "DÉCIMO": "10",
}

def normalizar_ciclo_ordinal(palabra: str) -> str:
    """Convierte 'PRIMER' → '01', 'SEGUNDO' → '02', etc."""
    clave = palabra.upper().replace("É", "E").replace("Ó", "O")
    return CICLO_MAP.get(clave, clave)


# ─────────────────────────────────────────────
# 4. PARSERS
# ─────────────────────────────────────────────

# ── 4a. UNALM – Estadística Informática ─────────────────────────────────

def parsear_unalm(texto: str) -> list[dict]:
    """
    Cabecera:
        ESPECIALIDAD: <carrera>
        CICLO: 01
    Fila de curso:
        EP1055   Administración General   Obligatorio   2  2  3
    El tipo de curso (Obligatorio/General/…) actúa como ancla para
    identificar el fin del nombre del curso.
    """
    RE_CARRERA = re.compile(r"ESPECIALIDAD:\s*(.+)", re.IGNORECASE)
    RE_CICLO   = re.compile(r"^CICLO:\s*(\d+)\s*$", re.IGNORECASE)
    PAT_TIPO   = r"(?:Obligatorio|General|Electivo[^0-9\n]*|Deportivo[^0-9\n]*)"
    RE_CURSO   = re.compile(
        rf"^({PAT_CODIGO}|DEP|ELC|PP\d{{4}})\s+",
        re.IGNORECASE,
    )
    # Columna donde empieza el campo TIPO (detectado del encabezado)
    COL_TIPO = 57

    registros = []
    universidad, carrera, ciclo = "UNALM", "", ""

    for raw in texto.splitlines():
        linea = raw.rstrip()
        if not linea.strip():
            continue

        m_car = RE_CARRERA.match(linea.strip())
        if m_car:
            carrera = m_car.group(1).strip().title()
            continue

        m_cic = RE_CICLO.match(linea.strip())
        if m_cic:
            ciclo = m_cic.group(1).zfill(2)
            continue

        m_cur = RE_CURSO.match(linea)
        if m_cur and ciclo:
            codigo = m_cur.group(1).strip()
            # Usar columna fija para separar nombre del tipo de curso
            nombre_raw = linea[:COL_TIPO] if len(linea) >= COL_TIPO else linea
            nombre_raw = re.sub(rf"^({PAT_CODIGO}|DEP|ELC|PP\d{{4}})\s+", "", nombre_raw, flags=re.IGNORECASE)
            nombre = limpiar_nombre_curso(nombre_raw)
            if not nombre:
                continue
            registros.append({
                "universidad": universidad,
                "carrera":     carrera,
                "ciclo":       ciclo,
                "codigo":      codigo,
                "curso":       nombre,
            })

    return registros


# ── 4b. UNI – Ingeniería de Sistemas ────────────────────────────────────

def parsear_uni_sistemas(texto: str) -> list[dict]:
    """
    Cabecera:
        UNIVERSIDAD NACIONAL DE INGENIERÍA
        PLAN DE ESTUDIOS INGENIERÍA DE SISTEMAS - 2018
        PRIMER CICLO:
    Fila de curso (layout con columnas fijas):
        ' FB101 Geometría Analítica                                   NINGUNO  2  2  4'
    Estrategia: capturar desde el código hasta el fin de línea,
    luego limpiar prereqs del nombre.
    """
    RE_CARRERA = re.compile(
        r"PLAN DE ESTUDIOS\s+(.+?)(?:\s*-\s*\d{4})?$", re.IGNORECASE
    )
    RE_CICLO   = re.compile(
        r"^(PRIMER|SEGUNDO|TERCER|CUARTO|QUINTO|SEXTO|S[EÉ]PTIMO|OCTAVO|NOVENO|D[EÉ]CIMO)\s+CICLO",
        re.IGNORECASE,
    )
    # Captura código + TODO lo que sigue en la línea
    RE_CURSO   = re.compile(rf"^\s*({PAT_CODIGO})\s+(.+?)\s*$")
    # Fila de total de horas: sólo números → ignorar
    RE_TOTAL   = re.compile(r"^\s*[\d\s]+$")
    # Columna donde inicia el campo PRE REQ en el layout (detectado del encabezado)
    COL_PREREQ = 61

    registros = []
    universidad = "Universidad Nacional de Ingeniería"
    carrera, ciclo = "", ""

    for raw in texto.splitlines():
        linea = raw.rstrip()
        if not linea.strip():
            continue

        # Encabezado de carrera
        m = RE_CARRERA.match(linea.strip())
        if m:
            candidato = m.group(1).strip().title()
            if len(candidato) > 4:
                carrera = candidato
            continue

        # Encabezado de ciclo
        m = RE_CICLO.match(linea.strip())
        if m:
            ciclo = normalizar_ciclo_ordinal(m.group(1))
            continue

        # Fila de totales → ignorar
        if RE_TOTAL.match(linea.strip()):
            continue

        # Curso: usar posición de columna para separar nombre de prereq
        m = RE_CURSO.match(linea)
        if m and ciclo:
            # Tomar solo hasta la columna COL_PREREQ para el nombre
            nombre_raw = linea[:COL_PREREQ] if len(linea) >= COL_PREREQ else linea
            # Quitar el código del comienzo para quedarnos solo con el nombre
            nombre_raw = re.sub(rf"^\s*{PAT_CODIGO}\s+", "", nombre_raw)
            nombre = limpiar_nombre_curso(nombre_raw)
            if not nombre or re.match(r"^(Nombre|Código|TOTAL|PRE REQ)", nombre, re.IGNORECASE):
                continue
            registros.append({
                "universidad": universidad,
                "carrera":     carrera,
                "ciclo":       ciclo,
                "codigo":      m.group(1).strip(),
                "curso":       nombre,
            })

    return registros


# ── 4c. UNI – Ingeniería de Software ────────────────────────────────────

def parsear_uni_software(texto: str) -> list[dict]:
    """
    Similar a Sistemas, pero sin dos puntos en el ciclo y con
    créditos antes de prereqs:
        'SW101  Introducción a la Ingeniería de Software  3  2  2  4  F  NINGUNO'
    """
    RE_CICLO = re.compile(
        r"^(PRIMER|SEGUNDO|TERCER|CUARTO|QUINTO|SEXTO|S[EÉ]PTIMO|OCTAVO|NOVENO|D[EÉ]CIMO)\s+CICLO\s*$",
        re.IGNORECASE,
    )
    # Nombre termina antes de 2+ espacios seguidos de dígito (créditos)
    RE_CURSO = re.compile(
        rf"^\s*({PAT_CODIGO})\s{{2,}}(.+?)\s{{2,}}\d"
    )

    registros = []
    universidad = "Universidad Nacional de Ingeniería"
    carrera, ciclo = "Ingeniería de Software", ""

    for raw in texto.splitlines():
        linea = raw.rstrip()
        if not linea.strip():
            continue

        m = RE_CICLO.match(linea.strip())
        if m:
            ciclo = normalizar_ciclo_ordinal(m.group(1))
            continue

        m = RE_CURSO.match(linea)
        if m and ciclo:
            nombre = limpiar_nombre_curso(m.group(2))
            if not nombre or re.match(r"^(Nombre|Código|TOTAL|CRÉDITO)", nombre, re.IGNORECASE):
                continue
            registros.append({
                "universidad": universidad,
                "carrera":     carrera,
                "ciclo":       ciclo,
                "codigo":      m.group(1).strip(),
                "curso":       nombre,
            })

    return registros


# ─────────────────────────────────────────────
# 5. PIPELINE PRINCIPAL
# ─────────────────────────────────────────────

PARSERS = {
    "unalm":        parsear_unalm,
    "uni_sistemas": parsear_uni_sistemas,
    "uni_software": parsear_uni_software,
}


def construir_dataframe(archivos: list[dict], pdf_dir: Path) -> pd.DataFrame:
    todos = []

    for item in archivos:
        ruta = pdf_dir / item["path"]
        print(f"\n📄 Procesando: {ruta.name}")

        texto   = extraer_texto_pdf(ruta)
        parser  = PARSERS.get(item["tipo"])

        if parser is None:
            print(f"  ⚠️  Tipo '{item['tipo']}' no reconocido, omitiendo.")
            continue

        registros = parser(texto)
        print(f"  ✅ {len(registros)} cursos extraídos.")
        todos.extend(registros)

    df2 = pd.DataFrame(todos, columns=["universidad", "carrera", "ciclo", "codigo", "curso"])

    # Limpieza final del DataFrame
    for col in df.columns:
        df2[col] = df2[col].str.strip()
    df2["ciclo"] = df2["ciclo"].str.zfill(2)

    # Quitar duplicados exactos (electivos que aparecen fuera de ciclo)
    df2 = df2.drop_duplicates().reset_index(drop=True)

    return df2


# ─────────────────────────────────────────────
# 6. EJECUCIÓN
# ─────────────────────────────────────────────

if __name__ == "__main__":
    df2 = construir_dataframe(ARCHIVOS, PDF_DIR)

    pd.set_option("display.max_rows", 300)
    pd.set_option("display.max_colwidth", 70)
    pd.set_option("display.width", 130)

    print("\n" + "=" * 90)
    print(f"DATAFRAME FINAL — {len(df2)} registros")
    print("=" * 90)
    print(df.to_string(index=True))

    print("\n── Resumen por universidad / carrera ──")
    resumen = (
        df2.groupby(["universidad", "carrera"])
          .agg(ciclos=("ciclo", "nunique"), n_cursos=("curso", "count"))
          .reset_index()
    )
    print(resumen.to_string(index=False))
    df = pd.concat([df, df2.remove(columns='codigo')], axis=0)
    # Exportar
    salida = Path("/mnt/user-data/outputs/planes_estudio.csv")
    df2.to_csv(salida, index=False, encoding="utf-8-sig")
    print(f"\n💾 CSV exportado: {salida}")


📄 Procesando: Plan de estudios Estadística Informática.pdf
  ✅ 0 cursos extraídos.

📄 Procesando: PLAN-ESTUDIOS-INGENIERIA-SISTEMAS-2018-v01_-_VERSION_01-pages-1-4.pdf
  ✅ 90 cursos extraídos.

📄 Procesando: PLAN_ESTUDIOS-_INGENIERIA-SOFTWARE_11-07-2021-CF-V1.pdf
  ✅ 1 cursos extraídos.

DATAFRAME FINAL — 91 registros
                  universidad                                                   carrera                                                                       curso  ciclo
0    Universidad del Pacífico                                    Ingeniería Empresarial                                                      Nivelación en Lenguaje    0.0
1    Universidad del Pacífico                                    Ingeniería Empresarial                                                   Nivelación en Matemáticas    0.0
2    Universidad del Pacífico                                    Ingeniería Empresarial                                                   Nivelación en Informática   

AttributeError: 'DataFrame' object has no attribute 'remove'

In [ ]:
df2 = df2.drop(columns='codigo')
df2


In [158]:
columns = ['universidad', 'carrera', 'curso', 'ciclo']
df3 = df2[columns]

In [161]:
df.to_csv('df_new_new.csv')

In [179]:
df = pd.read_csv('df_new_new.csv')

In [180]:
df = pd.concat([df, df3],axis=0)
df

,Unnamed: 0,universidad,carrera,curso,ciclo
0,0.0,Universidad del Pacífico,Ingeniería Empresarial,Nivelación en Lenguaje,0.0
1,1.0,Universidad del Pacífico,Ingeniería Empresarial,Nivelación en Matemáticas,0.0
2,2.0,Universidad del Pacífico,Ingeniería Empresarial,Nivelación en Informática,0.0
3,3.0,Universidad del Pacífico,Ingeniería Empresarial,Lenguaje I,1.0
4,4.0,Universidad del Pacífico,Ingeniería Empresarial,Matemáticas I,1.0
...,...,...,...,...,...
86,NaN,Universidad Nacional de Ingeniería,Ingeniería De Sistemas,Taller de Diseño de Sistemas Inteligentes,10
87,NaN,Universidad Nacional de Ingeniería,Ingeniería De Sistemas,Analítica de Datos FB405 2,10
88,NaN,Universidad Nacional de Ingeniería,Ingeniería De Sistemas,Ciberseguridad SI904 2,10
89,NaN,Universidad Nacional de Ingeniería,Ingeniería De Sistemas,Diseño Gráfico e Iconográfico SI302 1,10


In [181]:
df = pd.concat([df, df_total], axis=0)
df

,Unnamed: 0,universidad,carrera,curso,ciclo
0,0.0,Universidad del Pacífico,Ingeniería Empresarial,Nivelación en Lenguaje,0.0
1,1.0,Universidad del Pacífico,Ingeniería Empresarial,Nivelación en Matemáticas,0.0
2,2.0,Universidad del Pacífico,Ingeniería Empresarial,Nivelación en Informática,0.0
3,3.0,Universidad del Pacífico,Ingeniería Empresarial,Lenguaje I,1.0
4,4.0,Universidad del Pacífico,Ingeniería Empresarial,Matemáticas I,1.0
...,...,...,...,...,...
373,NaN,Universidad Peruana Cayetano Heredia (UPCH),Computación Científica,DERECHO Y ÉTICA EN LA INFORMÁTICA,NaN
374,NaN,Universidad Peruana Cayetano Heredia (UPCH),Ingeniería Informática,TÓPICOS AVANZADOS DE SEGURIDAD INFORMÁTICA,NaN
375,NaN,Universidad Peruana Cayetano Heredia (UPCH),Computación Científica,TÓPICOS AVANZADOS DE SEGURIDAD INFORMÁTICA,NaN
376,NaN,Universidad Peruana Cayetano Heredia (UPCH),Ingeniería Informática,ASIGNATURA ELECTIVA IV,NaN


In [182]:
df['universidad'].unique()

array(['Universidad del Pacífico', 'UPC', 'Científica del Sur', 'ESAN',
       'Universidad Nacional de Ingeniería',
       'Pontificia Universidad Catolica del Peru (PUCP)',
       'Universidad Peruana Cayetano Heredia (UPCH)'], dtype=object)

In [183]:
import pandas as pd

rows = []

def add(universidad, carrera, curso, ciclo):
    rows.append({"universidad": universidad, "carrera": carrera, "curso": curso, "ciclo": ciclo})

# =========================================================
# UNALM - ESTADÍSTICA INFORMÁTICA (Plan vigente 2019)
# =========================================================
U = "UNALM"
C = "Estadística Informática"

unalm_cursos = {
    1: ["Administración General", "Análisis Matemático I", "Curso de Deportes o Actividades Culturales",
        "Ecología General", "Introducción a la Ciencia de Datos", "Química General", "Sociedad y Cultura Peruana"],
    2: ["Análisis Matemático II", "Economía General", "Física General", "Herramientas de Gestión Empresarial",
        "Lenguaje y Comunicación", "Perú en el Contexto Internacional", "Técnicas de Exploración de Datos"],
    3: ["Algebra Matricial", "Estadística General", "Ingeniería de Procesos", "Lenguaje de Programación I",
        "Matemáticas Discretas", "Redacción y Argumentación"],
    4: ["Análisis Estadístico", "Ética y Ciudadanía", "Lenguaje de Programación II", "Metodología de la Investigación",
        "Métodos de Optimización", "Sistemas de Gestión de Base de Datos I"],
    5: ["Análisis de Regresión", "Calculo de Probabilidades", "Diseños Experimentales I", "Estrategias de Muestreo",
        "Lenguaje de Programación III"],
    6: ["Algoritmia", "Diseños Experimentales II", "Inferencia Estadística", "Sistemas de Gestión de Base de Datos II",
        "Técnicas Multivariadas"],
    7: ["Estadística Bayesiana", "Estadística Computacional", "Marketing", "Modelos Lineales I",
        "Sistemas de Información Gerencial"],
    8: ["Estadística no Paramétrica", "Gestión Estratégica de Datos", "Investigación de Mercados",
        "Máquinas de Aprendizaje", "Modelos Lineales II", "Seminario en Estadística Informática I"],
    9: ["Análisis de Series de Tiempo", "Ciencias de Datos I", "Curso Electivo de Carrera", "Estadística Espacial",
        "Gestión de Proyectos de Información", "Practicas Pre-Profesionales", "Seminario en Estadística Informática II"],
    10: ["Ciencias de Datos II", "Curso Electivo de Carrera", "Seminario en Estadística Informática III",
         "Tecnologías Emergentes", "Trabajo de investigación"],
}
for ciclo, cursos in unalm_cursos.items():
    for curso in cursos:
        add(U, C, curso, ciclo)

# Cursos electivos de carrera (sin ciclo fijo asignado en el plan)
for curso in ["Análisis de Sobrevivencia y Confiabilidad", "Control Estadístico de la Calidad",
              "Estadística Actuarial", "Métodos de Procesos Estocásticos", "Principios de Finanzas"]:
    add(U, C, curso, "Electivo")

# =========================================================
# ULIMA - INGENIERÍA DE SISTEMAS (malla 2026)
# =========================================================
U = "ULIMA"
C = "Ingeniería de Sistemas"

ulima_sistemas = {
    1: ["Introducción a la Ingeniería", "Metodologías de Investigación", "Precálculo",
        "Lenguaje y Comunicación I", "Procesos Psicológicos", "Estructuras Discretas de Computación",
        "Introducción a la Programación"],
    2: ["Cálculo I", "Álgebra Lineal", "Lenguaje y Comunicación II", "Ética Ciudadana",
        "Programación Orientada a Objetos", "Arquitectura de Computadoras"],
    3: ["Cálculo II", "Fundamentos de Economía", "Filosofía Aplicada", "Estructuras de Datos I",
        "Paradigmas de Programación", "Física para Sistemas"],
    4: ["Cálculo III", "Introducción al Comercio Internacional", "Estadística Aplicada",
        "Estructuras de Datos II", "Programación Web", "Modelamiento de Base de Datos"],
    5: ["Investigación de Operaciones I", "Ingeniería de Procesos de Negocio", "Sistemas Operativos",
        "Análisis y Diseño de Algoritmos", "Internet de las Cosas / Internet of Things", "Gestión de Base de Datos"],
    6: ["Redes de Computadoras", "Sistemas Organizacionales / Organizational Systems", "Simulación",
        "Ingeniería de Software I", "Seguridad de Sistemas", "Redes Avanzadas", "Costeo de Operaciones"],
    7: ["Gestión de Operaciones", "Modelación e Integración de Sistemas", "Auditoría y Control de Sistemas",
        "Ingeniería del Conocimiento", "Ciberseguridad / Cybersecurity", "Gestión Financiera",
        "Desarrollo de Competencias Gerenciales"],
    8: ["Sistemas ERP", "Ingeniería de Software II", "Deep Learning", "Programación Móvil",
        "Tópicos Avanzados en Ciberseguridad", "Sistemas Distribuidos", "Computación en la Nube / Cloud Computing"],
    9: ["Planeamiento Estratégico", "Aprendizaje de Máquina / Machine Learning", "Analítica con Big Data",
        "Analítica de Negocios", "Proyecto de Desarrollo de Software", "Innovación Digital",
        "Proyecto de Videojuegos", "Arquitectura de Software"],
    10: ["Gestión de Servicios Digitales", "Proyecto Integrador de Sistemas", "Gestión de Proyectos",
         "Arquitectura Empresarial", "Interacción Humano Computadora / Human Computer Interaction",
         "Arquitectura de Tecnologías de la Información", "DevOps"],
}
for ciclo, cursos in ulima_sistemas.items():
    for curso in cursos:
        add(U, C, curso, ciclo)

add(U, C, "Seguridad, Salud Ocupacional y Bienestar Organizacional", 1)
add(U, C, "Propuesta de Investigación", 9)
add(U, C, "Seminario de Investigación I", 9)
add(U, C, "Seminario de Investigación II", 10)
add(U, C, "Electivo", "Electivo")

# =========================================================
# ULIMA - INGENIERÍA INDUSTRIAL (malla 2026)
# =========================================================
U = "ULIMA"
C = "Ingeniería Industrial"

ulima_industrial = {
    1: ["Introducción a la Ingeniería", "Metodologías de Investigación", "Precálculo",
        "Lenguaje y Comunicación I", "Procesos Psicológicos", "Fundamentos de Programación",
        "Seguridad, Salud Ocupacional y Bienestar Organizacional"],
    2: ["Cálculo I", "Álgebra Lineal", "Lenguaje y Comunicación II", "Ética Ciudadana",
        "Electricidad y Electrónica", "Análisis de Problemas de Ingeniería"],
    3: ["Cálculo II", "Fundamentos de Economía", "Filosofía Aplicada", "Química General",
        "Física I", "Mecánica"],
    4: ["Cálculo III", "Introducción al Comercio Internacional", "Física II", "Ecuaciones Diferenciales",
        "Procesos de Manufactura", "Diseño Asistido por el Computador"],
    5: ["Investigación de Operaciones I", "Ingeniería Económica", "Costeo de Operaciones",
        "Diseño de Experimentos", "Materiales en la Manufactura", "Programación para Ingeniería"],
    6: ["Investigación de Operaciones II", "Fundamentos de Operaciones y Logística / Operations and Logistics Fundamentals",
        "Ergonomía y Diseño del Trabajo", "Procesos Industriales", "Automatización Industrial",
        "Introducción a Sistemas de Gestión de Bases de Datos", "Tecnologías de Programación"],
    7: ["Ingeniería Financiera", "Planeamiento y Control de Operaciones", "Diseño de Instalaciones",
        "Innovación en Ingeniería", "Calidad / Quality", "Inteligencia de Negocios",
        "Gestión de Proyectos de Diseño", "Diseño y Prototipado"],
    8: ["Simulación de Procesos", "Modelos de Sistemas Logísticos", "Modelamiento Predictivo de Datos",
        "Gestión de Proyectos", "Sistemas Integrados de Gestión", "Estrategia de Inteligencia Empresarial",
        "Transformación Digital", "Robotic Process Automation"],
    9: ["Ética y Gestión Humana", "Proyecto de Ingeniería Aplicada I", "Gerencia Estratégica / Strategic Management",
        "Supply Chain Management", "Compras y Gestión del Abastecimiento", "Formulación y Evaluación de Proyectos",
        "Metodologías Ágiles", "Machine Learning", "Sistemas de Información Gerencial"],
    10: ["Proyecto de Ingeniería Aplicada II", "Sistemas Organizacionales / Organizational Systems",
         "Ingeniería Comercial / Commercial Engineering", "Juego de Negocios", "Dirección en Implementación de Proyectos",
         "Gestión del Comercio Internacional", "Creatividad, Innovación y Emprendimiento",
         "Diseño de Proyectos Sostenibles", "Taller de Liderazgo", "Taller de Habilidades Gerenciales"],
}
for ciclo, cursos in ulima_industrial.items():
    for curso in cursos:
        add(U, C, curso, ciclo)

for curso in ["Ingeniería de Seguridad", "Procesos Logísticos ERP / ERP Process Workshop",
              "Ingeniería del Transporte y Distribución", "Gestión de Operaciones de Servicios",
              "Gestión de Recursos", "Gestión de Riesgos y Portafolios", "Gerencia B2B",
              "Herramientas de Marketing Digital", "Sostenibilidad Industrial", "Herramientas Informáticas",
              "Lean Six Sigma", "Electivo"]:
    add(U, C, curso, "Electivo")

# =========================================================
# UNMSM - INGENIERÍA DE SOFTWARE (Plan de Estudios 2023)
# =========================================================
U = "UNMSM"
C = "Ingeniería de Software"

unmsm_software = {
    1: ["Redacción y Técnicas de Comunicación Efectiva I", "Métodos de Estudios Universitarios",
        "Desarrollo Personal y Liderazgo", "Cálculo I", "Biología para Ciencias e Ingeniería",
        "Algebra y Geometría Analítica I", "Medio Ambiente y Desarrollo Sostenible"],
    2: ["Redacción y Técnicas de Comunicación Efectiva II", "Investigación Formativa",
        "Realidad Nacional y Mundial", "Cálculo II", "Física I", "Química General",
        "Introducción a las Ciencias e Ingeniería"],
    3: ["Algorítmica I", "Estadística y Probabilidades", "Física Eléctrónica", "Ingeniería Económica",
        "Introducción al Desarrollo de Software", "Matemática Básica"],
    4: ["Algorítmica II", "Contabilidad para la Gestión", "Organización y Administración",
        "Matemática Discreta", "Estructura de Datos", "Procesos de Software", "Sistemas Digitales"],
    5: ["Análisis y Diseño de Algoritmos", "Arquitectura de Computadoras", "Base de Datos I",
        "Computación Visual", "Economía para la Gestión", "Ingeniería de Requisitos",
        "Lenguajes y Compiladores"],
    6: ["Base de Datos II", "Calidad de Software", "Diseño de Software",
        "Etica Profesional y Legislación Informática", "Gestión de Proyecto de Software",
        "Interacción Hombre Computador", "Sistemas Operativos"],
    7: ["Arquitectura de Software", "Experiencia de Usuario y Usabilidad",
        "Gestión de la Configuración y Mantenimiento del Software", "Formación de Empresas de Software",
        "Inteligencia Artificial", "Pruebas de Software", "Redes y Transmisión de Datos"],
    8: ["Automatización y Control de Software", "Inteligencia de Negocios", "Metodología de la Investigación",
        "Minería de Datos", "Programación Concurrente y Paralela", "Taller de Construcción de Software Web",
        "Aseguramiento de la Calidad del Software"],
    9: ["Desarrollo de Tesis I", "Gestión de Riesgo del Software", "Gerencia de Tecnología de la Información",
        "Seguridad del Software", "Internet de las Cosas", "Taller de Construcción de Software Móvil",
        "Software Inteligente"],
    10: ["Analítica de Datos", "Desarrollo de Tesis II", "Práctica Pre Profesional",
         "Taller de Aplicaciones Sociales", "Innovación, Tecnología y Emprendimiento",
         "Tendencias en Ingeniería de Software"],
}
for ciclo, cursos in unmsm_software.items():
    for curso in cursos:
        add(U, C, curso, ciclo)

for curso in ["Programación y Computación", "Emprendimiento e Innovación",
              "Fundamentos de Riesgos de Desastres y Cambio Climático", "Taller de Música",
              "Taller de Danza", "Ingles Basico", "Seguridad e Higiene Ocupacional"]:
    add(U, C, curso, "Electivo")

# =========================================================
# UNMSM - INGENIERÍA DE SISTEMAS (Plan de Estudios 2023)
# =========================================================
U = "UNMSM"
C = "Ingeniería de Sistemas"

unmsm_sistemas = {
    1: ["Redacción y Técnicas de Comunicación Efectiva I", "Métodos de Estudio Universitario",
        "Desarrollo Personal y Liderazgo", "Cálculo I", "Biología para Ciencias e Ingeniería",
        "Álgebra y Geometría Analítica", "Medio Ambiente y Desarrollo Sostenible",
        "Programación y Computación"],
    2: ["Redacción y Técnicas de Comunicación Efectiva II", "Investigación Formativa",
        "Realidad Nacional y Mundial", "Cálculo II", "Física I", "Química General",
        "Introducción a las Ciencias e Ingeniería", "Emprendimiento e Innovación"],
    3: ["Introducción a la Computación", "Series y Ecuaciones Diferenciales", "Electromagnetismo y Óptica",
        "Programación de Computadoras I", "Fundamentos de Sistemas de Información", "Matemáticas Discretas"],
    4: ["Organización Empresarial", "Programación de Computadoras II", "Métodos Numéricos",
        "Arquitectura de Computadoras", "Estadística I", "Ingeniería Económica"],
    5: ["Análisis de Sistemas de Información", "Diseño de Base de Datos", "Estructura de Datos",
        "Estadística II", "Sistemas Operativos", "Investigación Operativa"],
    6: ["Diseño de Sistemas de Información", "Administración de Base de Datos", "Redes y Comunicaciones",
        "Diseño de Interfases de Usuario", "Modelos y Simulación", "Finanzas para la Gestión"],
    7: ["Interacción Humano Computador", "Desarrollo de Aplicaciones Web", "Inteligencia Artificial",
        "Inteligencia de Negocios", "Gestión de Datos Masivos", "Internet de las Cosas"],
    8: ["Computación Visual", "Desarrollo de Aplicaciones Móviles", "Sistemas Distribuidos",
        "Sistemas Inteligentes", "Gestión de Proyectos de TI", "Metodología de la Investigación"],
    9: ["Arquitectura Empresarial", "Gestión de Procesos de Negocio", "Minería de Datos",
        "Auditoría de Sistemas", "Seguridad de la Información", "Proyecto de Tesis I"],
    10: ["Gestión de Tecnologías de Información", "Arquitectura de Servicios",
         "Tendencias en Sistemas de Información", "Proyecto de Fin de Carrera",
         "Ética Profesional y Emprendimiento", "Proyecto de Tesis II"],
}
for ciclo, cursos in unmsm_sistemas.items():
    for curso in cursos:
        add(U, C, curso, ciclo)

# =========================================================
# UNMSM - CIENCIA DE LA COMPUTACIÓN (Plan de Estudios 2023)
# =========================================================
U = "UNMSM"
C = "Ciencia de la Computación"

unmsm_ccomp = {
    1: ["Redacción y Técnicas de Comunicación Efectiva I", "Métodos de Estudio Universitario",
        "Desarrollo Personal y Liderazgo", "Cálculo I", "Biología para Ciencias e Ingeniería",
        "Álgebra y Geometría Analítica", "Medio Ambiente y Desarrollo Sostenible"],
    2: ["Redacción y Técnicas de Comunicación Efectiva II", "Investigación Formativa",
        "Realidad Nacional y Mundial", "Cálculo II", "Física I", "Química General",
        "Introducción a las Ciencias e Ingeniería", "Emprendimiento e Innovación"],
    3: ["Introducción a la Ciencia de la Computación", "Programación de Computadoras I",
        "Desarrollo Basado en Plataformas", "Estructuras Discretas", "Series y Ecuaciones Diferenciales",
        "Ingeniería Económica", "Óptica y Electro-Magnetismo"],
    4: ["Algoritmos y Estructuras de Datos", "Teoría de la Computación", "Base de Datos I",
        "Arquitectura de Computadores", "Estadística y Probabilidades", "Análisis Numérico"],
    5: ["Análisis y Diseño de Algoritmos", "Base de Datos II", "Ingeniería de Software I",
        "Sistemas Operativos", "Programación de Computadoras II", "Investigación Operativa"],
    6: ["Ingeniería de Software II", "Redes y Comunicaciones", "Estructura de Datos Avanzado",
        "Tópicos de Sistemas Operativos Avanzados", "Lenguajes y Compiladores I", "Modelos y Simulación",
        "Procesos Estocásticos"],
    7: ["Interacción Humano Computador", "Lenguajes y Compiladores II", "Computación Paralela y Distribuida",
        "Computación Grafica", "Heurísticas y Meta Heurísticas", "Investigación e Innovación"],
    8: ["Seguridad en Computación", "Tópicos en Ciencia de Datos", "Inteligencia Artificial",
        "Arte y Tecnología", "Proyecto Final de Carrera I", "Computación Verde"],
    9: ["Tópicos en Inteligencia Artificial", "Practica Preprofesional I", "Tópicos en Computación Grafica",
        "Proyecto Final de Carrera II"],
    10: ["Practica Preprofesional II", "Tecnología Basada en Internet", "Bioinformática y Bioestadística",
         "Proyecto Final de Carrera III"],
}
for ciclo, cursos in unmsm_ccomp.items():
    for curso in cursos:
        add(U, C, curso, ciclo)

for curso in ["Proceso Cultural Andino", "Programación y Computación", "Dibujo Técnico",
              "Inglés para Escritura Académica", "Matlab", "Cálculos Básicos en Química",
              "Seguridad e Higiene en Laboratorio", "Fundamentos de Riesgos de Desastres y Cambio Climático",
              "Geografia Económica del Perú", "Ciudadanía y Derechos Fundamentales",
              "Taller de Electricidad y Electronica", "Economia General", "Emprendimiento e Innovación",
              "Taller de Música", "Taller de Danza", "Apreciación de Cine", "Quechua"]:
    add(U, C, curso, "Electivo")

# =========================================================
# UNMSM - COMPUTACIÓN CIENTÍFICA (Plan de Estudios 2018)
# =========================================================
U = "UNMSM"
C = "Computación Científica"

unmsm_ccientifica = {
    1: ["Lenguaje", "Métodos de Estudio Universitario", "Gestión Personal", "Cálculo I",
        "Matemática Básica", "Biología", "Electivo"],
    2: ["Fundamentos de Investigación Científica", "Medio Ambiente y Desarrollo Sostenible",
        "Realidad Nacional y Mundial", "Cálculo II", "Química Inorgánica y Orgánica", "Física General",
        "Electivo"],
    3: ["Cálculo III", "Elementos de Álgebra y Geometría", "Algorítmica y Fundamentos de Programación",
        "Elementos de Matemática Discreta y Computacional", "Inglés I"],
    4: ["Cálculo IV", "Álgebra", "Modelos Matemáticos de la Física", "Estructura de Datos", "Inglés II"],
    5: ["Análisis Real", "Ecuaciones Diferenciales Ordinarias", "Base de Datos e Ingeniería de Software",
        "Métodos Numéricos y Programación I", "Inglés III"],
    6: ["Análisis Funcional I", "Probabilidad e Inferencia Estadística", "Métodos Numéricos y Programación II",
        "Sistemas Operativos", "Técnicas de Modelamiento"],
    7: ["Análisis Funcional II", "Computación Gráfica", "Modelos Matemáticos de la Ciencia I",
        "Inteligencia Artificial", "Electivo"],
    8: ["Ecuaciones Diferenciales Parciales I", "Modelos Matemáticos de la Ciencia II",
        "Redes, Arquitectura y Comunicaciones", "Proyecto de Investigación", "Electivo"],
    9: ["Optimización", "Modelos Matemáticos de la Ciencia III", "Métodos Numéricos y Programación III",
        "Desarrollo del Proyecto de Investigación", "Electivo"],
    10: ["Modelos Matemáticos de la Ciencia IV", "Formulación, Evaluación y Gestión de Proyectos",
         "Práctica Pre Profesional", "Presentación y Sustentación del Proyecto de Investigación", "Electivo"],
}
for ciclo, cursos in unmsm_ccientifica.items():
    for curso in cursos:
        add(U, C, curso, ciclo)

for curso in ["Ecuaciones Diferenciales Parciales II", "Análisis Complejo y Aplicado", "Ecuaciones Integrales",
              "Análisis Complejo en Cn", "Problemas Inversos", "Sistemas Dinámicos", "Meteorología",
              "Termoelasticidad", "Mecánica Cuántica", "Geofísica", "Modelamiento para la Ecología",
              "Método de los Elementos Finitos", "Optimización Dinámica y Convexa", "Electrónica Digital",
              "Sistemas Inteligentes", "Sistemas de Información Gerencial",
              "Fundamentos Avanzados de Computación", "Video Juegos y Aplicaciones Móviles",
              "Programación Segura en Sistemas Computacionales", "Introducción a Machine Learning"]:
    add(U, C, curso, "Electivo")

# =========================================================
# Construcción del DataFrame final
# =========================================================
df4 = pd.DataFrame(rows, columns=["universidad", "carrera", "curso", "ciclo"])


In [184]:
df4

,universidad,carrera,curso,ciclo
0,UNALM,Estadística Informática,Administración General,1
1,UNALM,Estadística Informática,Análisis Matemático I,1
2,UNALM,Estadística Informática,Curso de Deportes o Actividades Culturales,1
3,UNALM,Estadística Informática,Ecología General,1
4,UNALM,Estadística Informática,Introducción a la Ciencia de Datos,1
...,...,...,...,...
507,UNMSM,Computación Científica,Sistemas de Información Gerencial,Electivo
508,UNMSM,Computación Científica,Fundamentos Avanzados de Computación,Electivo
509,UNMSM,Computación Científica,Video Juegos y Aplicaciones Móviles,Electivo
510,UNMSM,Computación Científica,Programación Segura en Sistemas Computacionales,Electivo


In [185]:
df = df.drop(columns='Unnamed: 0')

In [186]:
df = pd.concat([df, df4], axis=0)
df

,universidad,carrera,curso,ciclo
0,Universidad del Pacífico,Ingeniería Empresarial,Nivelación en Lenguaje,0.0
1,Universidad del Pacífico,Ingeniería Empresarial,Nivelación en Matemáticas,0.0
2,Universidad del Pacífico,Ingeniería Empresarial,Nivelación en Informática,0.0
3,Universidad del Pacífico,Ingeniería Empresarial,Lenguaje I,1.0
4,Universidad del Pacífico,Ingeniería Empresarial,Matemáticas I,1.0
...,...,...,...,...
507,UNMSM,Computación Científica,Sistemas de Información Gerencial,Electivo
508,UNMSM,Computación Científica,Fundamentos Avanzados de Computación,Electivo
509,UNMSM,Computación Científica,Video Juegos y Aplicaciones Móviles,Electivo
510,UNMSM,Computación Científica,Programación Segura en Sistemas Computacionales,Electivo


In [187]:
df['universidad'].unique()

array(['Universidad del Pacífico', 'UPC', 'Científica del Sur', 'ESAN',
       'Universidad Nacional de Ingeniería',
       'Pontificia Universidad Catolica del Peru (PUCP)',
       'Universidad Peruana Cayetano Heredia (UPCH)', 'UNALM', 'ULIMA',
       'UNMSM'], dtype=object)

In [194]:
import re
import unicodedata
import pandas as pd

df = pd.read_csv("/mnt/user-data/outputs/mallas_curriculares.csv")

# ---------------------------------------------------------
# Stopwords en español (lista propia, sin dependencias externas)
# ---------------------------------------------------------
STOPWORDS_ES = {
    "a", "al", "algo", "algunas", "algunos", "ante", "antes", "como", "con", "contra",
    "cual", "cuales", "cuando", "de", "del", "desde", "donde", "durante", "e", "el",
    "ella", "ellas", "ello", "ellos", "en", "entre", "era", "erais", "eran", "eres",
    "es", "esa", "esas", "ese", "eso", "esos", "esta", "estaba", "estado", "estamos",
    "estan", "estar", "estas", "este", "esto", "estos", "estoy", "fue", "fueron",
    "fui", "fuimos", "ha", "hace", "hacia", "han", "hasta", "hay", "la", "las", "le",
    "les", "lo", "los", "mas", "me", "mi", "mis", "mucho", "muchos", "muy", "nada",
    "ni", "no", "nos", "nosotros", "nuestra", "nuestras", "nuestro", "nuestros", "o",
    "os", "otra", "otras", "otro", "otros", "para", "pero", "poco", "por", "porque",
    "que", "quien", "quienes", "se", "sea", "segun", "ser", "si", "sin", "sobre",
    "somos", "son", "soy", "su", "sus", "tambien", "tanto", "te", "tener", "tiene",
    "tienen", "todo", "todos", "tu", "tus", "un", "una", "uno", "unos", "vosotros",
    "vuestra", "vuestras", "vuestro", "vuestros", "y", "ya", "yo",
}

# Stopwords en inglés (lista propia, sin dependencias externas) — para los
# nombres bilingües de cursos como "Strategic Management" / "Operations and Logistics"
STOPWORDS_EN = {
    "a", "about", "above", "after", "again", "against", "all", "am", "an", "and",
    "any", "are", "as", "at", "be", "because", "been", "before", "being", "below",
    "between", "both", "but", "by", "could", "did", "do", "does", "doing", "down",
    "during", "each", "few", "for", "from", "further", "had", "has", "have", "having",
    "he", "her", "here", "hers", "herself", "him", "himself", "his", "how", "i", "if",
    "in", "into", "is", "it", "its", "itself", "just", "me", "more", "most", "my",
    "myself", "no", "nor", "not", "now", "of", "off", "on", "once", "only", "or",
    "other", "our", "ours", "ourselves", "out", "over", "own", "same", "she", "should",
    "so", "some", "such", "than", "that", "the", "their", "theirs", "them",
    "themselves", "then", "there", "these", "they", "this", "those", "through", "to",
    "too", "under", "until", "up", "very", "was", "we", "were", "what", "when",
    "where", "which", "while", "who", "whom", "why", "will", "with", "you", "your",
    "yours", "yourself", "yourselves",
}

STOPWORDS = STOPWORDS_ES | STOPWORDS_EN

# ---------------------------------------------------------
# Detección de números romanos (I, II, III, IV, V, VI, VII, VIII, IX, X, ...)
# ---------------------------------------------------------
ROMANO_RE = re.compile(
    r"^(?=[mdclxvi])m{0,4}(cm|cd|d?c{0,3})(xc|xl|l?x{0,3})(ix|iv|v?i{0,3})$"
)


def es_numero_romano(token: str) -> bool:
    """Devuelve True si el token (en minúsculas, solo letras) es un número romano válido."""
    t = token.lower()
    if t == "" or not re.fullmatch(r"[mdclxvi]+", t):
        return False
    return bool(ROMANO_RE.match(t))


def quitar_acentos(texto: str) -> str:
    """Normaliza tildes (á->a, é->e, etc.) conservando la ñ."""
    nfkd = unicodedata.normalize("NFKD", texto)
    return "".join(c for c in nfkd if unicodedata.category(c) != "Mn")


def limpiar_texto(texto: str) -> str:
    if not isinstance(texto, str):
        return ""

    texto = quitar_acentos(texto)

    # Reemplaza cualquier caracter que no sea letra (incluye ñ) o espacio por espacio
    # (esto ya elimina signos de puntuación, "/", numeros arabigos, etc.)
    texto = re.sub(r"[^a-zA-Zñ\s]", " ", texto)

    # Tokeniza por espacios
    tokens = texto.split()

    tokens_limpios = []
    for tok in tokens:
        tok_lower = tok.lower()
        if tok_lower in STOPWORDS:
            continue
        if es_numero_romano(tok_lower):
            continue
        tokens_limpios.append(tok_lower)

    return " ".join(tokens_limpios)


# ---------------------------------------------------------
# Construcción de columnas nuevas
# ---------------------------------------------------------
df["texto_limpio"] = df["curso"].apply(limpiar_texto)
df["tokens"] = [texto.split() for texto in df["texto_limpio"]]

if __name__ == "__main__":
    pd.set_option("display.max_colwidth", 60)
    print(df[["curso", "texto_limpio", "tokens"]].sample(15, random_state=1).to_string(index=False))
    df.to_csv("/mnt/user-data/outputs/mallas_curriculares.csv", index=False, encoding="utf-8-sig")
    print("\nCSV actualizado y guardado.")

                                                                         curso                                                        texto_limpio                                                                     tokens
                                                  Análisis de Series de Tiempo                                              analisis series tiempo                                                 [analisis, series, tiempo]
                                                         Sistemas Inteligentes                                               sistemas inteligentes                                                   [sistemas, inteligentes]
                                                        Desarrollo de Tesis II                                                    desarrollo tesis                                                        [desarrollo, tesis]
                               Redacción y Técnicas de Comunicación Efectiva I                            redacc

In [195]:
df

,universidad,carrera,curso,ciclo,texto_limpio,tokens
0,UNALM,Estadística Informática,Administración General,1,administracion general,"[administracion, general]"
1,UNALM,Estadística Informática,Análisis Matemático I,1,analisis matematico,"[analisis, matematico]"
2,UNALM,Estadística Informática,Curso de Deportes o Actividades Culturales,1,curso deportes actividades culturales,"[curso, deportes, actividades, culturales]"
3,UNALM,Estadística Informática,Ecología General,1,ecologia general,"[ecologia, general]"
4,UNALM,Estadística Informática,Introducción a la Ciencia de Datos,1,introduccion ciencia datos,"[introduccion, ciencia, datos]"
...,...,...,...,...,...,...
507,UNMSM,Computación Científica,Sistemas de Información Gerencial,Electivo,sistemas informacion gerencial,"[sistemas, informacion, gerencial]"
508,UNMSM,Computación Científica,Fundamentos Avanzados de Computación,Electivo,fundamentos avanzados computacion,"[fundamentos, avanzados, computacion]"
509,UNMSM,Computación Científica,Video Juegos y Aplicaciones Móviles,Electivo,video juegos aplicaciones moviles,"[video, juegos, aplicaciones, moviles]"
510,UNMSM,Computación Científica,Programación Segura en Sistemas Computacionales,Electivo,programacion segura sistemas computacionales,"[programacion, segura, sistemas, computacionales]"


In [196]:
type(df['tokens'][0][0])

str

In [197]:
df.to_csv('df_mallas_curriculares.csv')